In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/pattern_recognition")

!pip install poetry
!poetry install

import sys
sys.path.append("/root/.cache/pypoetry/virtualenvs/pattern-recognition-TH3-wNKi-py3.10/lib/python3.10/site-packages")

Mounted at /content/drive


In [ ]:
sys.path.append("/root/.cache/pypoetry/virtualenvs/pattern-recognition-nWrbh8gE-py3.10/lib/python3.10/site-packages")

In [ ]:
sys.path.append('/content/drive/MyDrive/pattern_recognition/src')

from data import GraphMatrixDataset, CNNMatrixDataset
from utils import P300Getter, train_model, plot_sample, show_progress, validate_model, infer_model
from interpretation import *
from models_cnn import *
from models_gnn import *
from graph import get_delaunay_graph, get_pos_init_graph, plot_graph, get_neighbors_graph

In [ ]:
import mne
import pandas as pd
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn, optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F

from scipy.spatial import Delaunay
import networkx as nx
import scipy.sparse as sp
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from statsmodels.stats.proportion import proportion_confint

from mne import Epochs, pick_types, events_from_annotations
from mne.channels import make_standard_montage, DigMontage
from mne.io import concatenate_raws, read_raw_edf
from mne.datasets import eegbci
from mne.decoding import Scaler

from torch_geometric.data import Data, InMemoryDataset

In [ ]:
def standardize_per_sample(X, eps=1e-8):
    """
    Standardize each sample (row) of X to zero mean and unit variance.
    X: array-like of shape (n_samples, n_timestamps)
    """
    X = np.asarray(X, dtype=np.float32)
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True)  # population std (ddof=0)
    return (X - mean) / np.maximum(std, eps)

In [ ]:
def positive_upsample(data, labels, ratio=16):
    positive_samples = data[labels == 1]
    positive_labels = labels[labels == 1]
    for i in range(ratio - 1):
        data = torch.cat([data, positive_samples])
        labels = torch.cat([labels, positive_labels])
    return data, labels

In [ ]:
@torch.no_grad()
def borderline_smote_torch(
    X: torch.Tensor,
    y: torch.Tensor,
    minority_label: int = 1,
    k: int = 5,          # minority neighbors for synthesis
    m: int = 10,         # neighbors for "danger" detection
    ratio: float = 1.0,  # desired minority after oversampling
    generator: torch.Generator | None = None,
    standardize_for_neighbors: bool = True,
    eps: float = 1e-6
):
    """
    Parameters
    - X: (n_samples, n_features) float tensor
    - y: (n_samples,) 1D tensor with binary labels (minority_label vs others)
    - minority_label: label value considered minority (default 1)
    - k: number of nearest minority neighbors used to synthesize (default 5)
    - m: number of nearest overall neighbors to detect "danger" (default 10)
    - ratio: target minority:majority ratio after oversampling (1.0 => equalize)
    - generator: torch.Generator for reproducibility (optional)
    - standardize_for_neighbors: z-score features for distance computations only
    - eps: small constant to avoid division by zero in std

    Returns
    - X_aug: (n_samples + n_synth, n_features) tensor (same dtype/device as X)
    - y_aug: (n_samples + n_synth,) tensor (same dtype/device as y)
    """
    device = X.device
    dtype = X.dtype

    # Ensure y is a 1D tensor
    y = y.view(-1)
    n, d = X.shape

    # Indices for classes
    y_bool_min = (y == minority_label)
    idx_min = torch.nonzero(y_bool_min, as_tuple=False).view(-1)
    idx_maj = torch.nonzero(~y_bool_min, as_tuple=False).view(-1)

    n_min = idx_min.numel()
    n_maj = idx_maj.numel()

    if n_min == 0 or n_maj == 0:
        return X, y  # nothing to do

    # Determine how many synthetic samples to generate
    target_n_min = math.ceil(ratio * n_maj)
    num_new = max(0, target_n_min - n_min)
    if num_new == 0:
        return X, y

    # Optionally standardize for neighbor search (but generate in original space)
    if standardize_for_neighbors:
        mean = X.mean(dim=0, keepdim=True)
        std = X.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
        X_std = (X - mean) / std
    else:
        X_std = X

    X_min = X[idx_min]
    X_min_std = X_std[idx_min]

    # Danger detection: find m nearest overall neighbors to each minority sample
    # Compute distances from minority to all samples in standardized space
    m_neighbors = min(m, max(1, n - 1))
    dist_all = torch.cdist(X_min_std, X_std, p=2)  # (n_min, n)
    # Get m+1 smallest to include self, then drop self
    topk_k = min(m_neighbors + 1, n)
    topk_vals, topk_idx = torch.topk(dist_all, k=topk_k, largest=False, dim=1)

    # Remove self index per row
    # Build filtered neighbor indices excluding self, keep up to m_neighbors
    neigh_idx_list = []
    for row in range(n_min):
        row_idx = topk_idx[row]
        # Exclude self (index equals original global index)
        self_idx = idx_min[row]
        mask = row_idx != self_idx
        filtered = row_idx[mask]
        neigh_idx_list.append(filtered[:m_neighbors])

    # Stack into tensor (pad if some rows shorter, but we limited to m_neighbors so lengths are equal)
    neigh_idx = torch.stack(neigh_idx_list, dim=0)  # (n_min, m_neighbors)

    # Count majority labels among neighbors
    neighbor_labels = y[neigh_idx]  # (n_min, m_neighbors)
    maj_counts = (neighbor_labels != minority_label).sum(dim=1)

    # Borderline-1 "DANGER" criterion: more than half neighbors are majority, but not all
    half = math.ceil(m_neighbors / 2)
    danger_mask = (maj_counts >= half) & (maj_counts < m_neighbors)
    idx_danger_min = idx_min[danger_mask]

    # Fallback: if no danger points, use all minority points
    if idx_danger_min.numel() == 0:
        idx_danger_min = idx_min

    # Prepare minority-only neighbor search for synthesis
    # For each danger point, find k nearest minority neighbors (excluding self)
    X_danger_std = X_std[idx_danger_min]
    dist_min = torch.cdist(X_danger_std, X_min_std, p=2)  # (n_danger, n_min)
    k_neighbors = min(k, max(1, n_min - 1))
    # Get k+1 to include potential self
    topk_vals_min, topk_idx_min = torch.topk(dist_min, k=min(k_neighbors + 1, n_min),
                                             largest=False, dim=1)
    # Remove self per row and keep k_neighbors
    neigh_min_idx_list = []
    for row in range(idx_danger_min.numel()):
        row_idx = topk_idx_min[row]
        # map minority-local indices to global indices
        global_min_indices = idx_min[row_idx]
        self_global = idx_danger_min[row]
        mask = global_min_indices != self_global
        filtered = global_min_indices[mask]
        neigh_min_idx_list.append(filtered[:k_neighbors])

    neigh_min_idx = torch.stack(neigh_min_idx_list, dim=0)  # (n_danger, k_neighbors)

    # Distribute num_new across danger points
    n_danger = idx_danger_min.numel()
    base = num_new // n_danger
    rem = num_new - base * n_danger
    counts = torch.full((n_danger,), base, dtype=torch.long, device=device)
    if rem > 0:
        if generator is None:
            sel = torch.randperm(n_danger, device=device)[:rem]
        else:
            sel = torch.randperm(n_danger, generator=generator, device=device)[:rem]
        counts[sel] += 1

    # Generate synthetic samples by interpolation in original space
    synth_list = []
    # Prepare random source
    def rand_uniform(shape):
        if generator is None:
            return torch.rand(shape, device=device, dtype=dtype)
        else:
            return torch.rand(shape, generator=generator, device=device, dtype=dtype)

    for row in range(n_danger):
        c = counts[row].item()
        if c <= 0:
            continue
        xi = X[idx_danger_min[row]]  # original space
        neighbors_global = neigh_min_idx[row]
        if neighbors_global.numel() == 0:
            continue
        # choose random neighbors for each synthetic sample
        if generator is None:
            choice = torch.randint(low=0, high=neighbors_global.numel(), size=(c,), device=device)
        else:
            choice = torch.randint(low=0, high=neighbors_global.numel(), size=(c,), device=device, generator=generator)
        xj = X[neighbors_global[choice]]  # (c, d)
        r = rand_uniform((c, 1))  # (c, 1)
        xi_rep = xi.unsqueeze(0).expand(c, -1)  # (c, d)
        x_new = xi_rep + r * (xj - xi_rep)      # (c, d)
        synth_list.append(x_new)

    if len(synth_list) == 0:
        return X, y

    X_syn = torch.cat(synth_list, dim=0).to(device=device, dtype=dtype)
    y_syn = torch.full((X_syn.size(0),), fill_value=minority_label, dtype=y.dtype, device=device)

    X_aug = torch.cat([X, X_syn], dim=0)
    y_aug = torch.cat([y, y_syn], dim=0)
    return X_aug, y_aug

In [ ]:
import glob

DATA_PATH = '/content/drive/MyDrive/pattern_recognition/processed_data/'

data = dict()
labels = dict()
for file in glob.glob(DATA_PATH + 'S*_P300_PZ.csv'):
    subj = file.split('/')[-1][:5]
    mat_data = np.loadtxt(file, delimiter=',')
    data[subj] = standardize_per_sample(mat_data[:, 1:])
    labels[subj] = mat_data[:, 0]
    data[subj] = np.vstack(data[subj])
    labels[subj] = np.hstack(labels[subj])

In [ ]:
min_len = min(arr.shape[0] for arr in labels.values())

for subj in data.keys():
    permutation = torch.randperm(min_len)
    data[subj] = torch.tensor(data[subj][:min_len]).float()[permutation]
    labels[subj] = torch.tensor(labels[subj].squeeze()[:min_len])[permutation]

In [ ]:
def build_multichannel_subject_dataset_unique(
    data, labels, n_channels, seed=None
):
    """
    data: Tensor [T, ...]
    labels: Tensor [T]
    return:
        X: Tensor [T', N, ...]
        y: Tensor [T']
    """

    if seed is not None:
        torch.manual_seed(seed)

    X_out = []
    y_out = []

    for cls in [1, 0]:
        idx = torch.where(labels == cls)[0]
        X_cls = data[idx]  # [M, ...]

        M = len(X_cls)
        usable = (M // n_channels) * n_channels
        if usable == 0:
            raise ValueError(
                f"Not enough samples for class {cls}: "
                f"{M} < {n_channels}"
            )

        # один shuffle
        perm = torch.randperm(M)
        X_cls = X_cls[perm][:usable]

        # режем без перекрытий
        chunks = X_cls.view(
            usable // n_channels,
            n_channels,
            *X_cls.shape[1:]
        )

        X_out.append(chunks)
        y_out.append(
            torch.full(
                (usable // n_channels,),
                cls,
                dtype=labels.dtype
            )
        )

    X_out = torch.cat(X_out, dim=0)
    y_out = torch.cat(y_out, dim=0)

    return X_out, y_out

In [ ]:
def multichannel_to_single_channel(X):
    """
    X: Tensor [T, N, H, W]
    return: Tensor [T, 1, H, W]
    """
    return X.mean(dim=1, keepdim=True)

## Эксперименты для 5 каналов

In [ ]:
N_CHANNELS = 5  # или 10

dataloaders = {}

for subj in data.keys():
    train_data, val_data, train_labels, val_labels = train_test_split(
        data[subj],
        labels[subj],
        test_size=0.15,
        shuffle=False
    )

    train_data, train_labels = borderline_smote_torch(
        train_data, train_labels,
        k=15, m=10,
        standardize_for_neighbors=False
    )

    X_train, y_train = build_multichannel_subject_dataset_unique(
        train_data, train_labels, n_channels=N_CHANNELS
    )
    X_val, y_val = build_multichannel_subject_dataset_unique(
        val_data, val_labels, n_channels=N_CHANNELS
    )

    train_dataset = CNNMatrixDataset(
        tensors=(X_train, y_train), with_target=True
    )
    val_dataset = CNNMatrixDataset(
        tensors=(X_val, y_val), with_target=True
    )

    dataloaders[subj] = {
        'train': DataLoader(train_dataset, batch_size=1024, shuffle=True),
        'val': DataLoader(val_dataset, batch_size=1024, shuffle=True),
    }

In [ ]:
mc_dataloaders = {}
single_dataloaders = {}

for subj, dataloader in dataloaders.items():
    # raw датасет уже multi-trial
    X_train, y_train = dataloader['train'].dataset.tensors
    X_val, y_val = dataloader['val'].dataset.tensors

    # single-channel через усреднение
    X_train_single = multichannel_to_single_channel(X_train)
    X_val_single = multichannel_to_single_channel(X_val)

    mc_train_ds = CNNMatrixDataset(
        tensors=(X_train, y_train), with_target=True
    )
    mc_val_ds = CNNMatrixDataset(
        tensors=(X_val, y_val), with_target=True
    )

    single_train_ds = CNNMatrixDataset(
        tensors=(X_train_single, y_train), with_target=True
    )
    single_val_ds = CNNMatrixDataset(
        tensors=(X_val_single, y_val), with_target=True
    )

    mc_dataloaders[subj] = {
        'train': DataLoader(mc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(mc_val_ds, batch_size=256, shuffle=True)
    }

    single_dataloaders[subj] = {
        'train': DataLoader(single_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(single_val_ds, batch_size=256, shuffle=True)
    }

### EEGNet, 5 каналов (weight_decay=1e-5)

MC: `EEGNet(250, 5, F1=128, D=1, F2=256)` — 5 стэкнутых эпох как каналы.  
SC: `EEGNet(250, 1, F1=8, D=8, F2=8)` — 5 эпох усреднены в одну.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_eegnet_5ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():

    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = EEGNet(
        250, 5, F1=128, D=1, F2=256
    ).to(my_device)

    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj],
        mc_dataloaders[subj],
        criterion,
        learning_params_mc,
        device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = EEGNet(
        250, 1, F1=8, D=8, F2=8
    ).to(my_device)

    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj],
        single_dataloaders[subj],
        criterion,
        learning_params_single,
        device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_eegnet_5ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (
            acc_mc['Min Accuracy'][-1],
            acc_mc['Max Accuracy'][-1]
        ),
        'sc_ci': (
            acc_sc['Min Accuracy'][-1],
            acc_sc['Max Accuracy'][-1]
        )
    }


===== Subject S1201 =====
Training complete in 3m 25s
Training complete in 1m 4s
MC — Acc: 0.906, ITR: 0.5501, CI: [0.866, 0.946]
SC — Acc: 0.921, ITR: 0.6006, CI: [0.884, 0.958]

===== Subject S0201 =====
Training complete in 3m 15s
Training complete in 1m 4s
MC — Acc: 0.901, ITR: 0.5342, CI: [0.860, 0.942]
SC — Acc: 0.861, ITR: 0.4194, CI: [0.814, 0.909]

===== Subject S0701 =====
Training complete in 3m 16s
Training complete in 1m 4s
MC — Acc: 0.817, ITR: 0.3130, CI: [0.763, 0.870]
SC — Acc: 0.807, ITR: 0.2922, CI: [0.752, 0.861]

===== Subject S0601 =====
Training complete in 3m 16s
Training complete in 1m 4s
MC — Acc: 0.787, ITR: 0.2531, CI: [0.731, 0.844]
SC — Acc: 0.807, ITR: 0.2922, CI: [0.752, 0.861]

===== Subject S1901 =====
Training complete in 3m 15s
Training complete in 1m 3s
MC — Acc: 0.970, ITR: 0.8071, CI: [0.947, 0.994]
SC — Acc: 0.970, ITR: 0.8071, CI: [0.947, 0.994]

===== Subject S2001 =====
Training complete in 3m 15s
Training complete in 1m 2s
MC — Acc: 0.792, I

In [ ]:
import pandas as pd
import numpy as np

rows = []

for subj, res in res_eegnet_5ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],

        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'MC Acc CI low': res['mc_ci'][0],
        'MC Acc CI high': res['mc_ci'][1],

        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
        'SC Acc CI low': res['sc_ci'][0],
        'SC Acc CI high': res['sc_ci'][1],
    })

df = pd.DataFrame(rows).sort_values('Subject')

summary = {
    'Subject': 'Mean ± Std',
    'Val size': df['Val size'].sum(),

    'MC Accuracy': f"{df['MC Accuracy'].mean():.3f} ± {df['MC Accuracy'].std():.3f}",
    'MC F1': f"{df['MC F1'].mean():.3f} ± {df['MC F1'].std():.3f}",
    'MC ITR': f"{df['MC ITR'].mean():.3f} ± {df['MC ITR'].std():.3f}",
    'SC Accuracy': f"{df['SC Accuracy'].mean():.3f} ± {df['SC Accuracy'].std():.3f}",
    'SC F1': f"{df['SC F1'].mean():.3f} ± {df['SC F1'].std():.3f}",
    'SC ITR': f"{df['SC ITR'].mean():.3f} ± {df['SC ITR'].std():.3f}",
    'MC Acc CI low': '',
    'MC Acc CI high': '',
    'SC Acc CI low': '',
    'SC Acc CI high': ''
}

df_summary = pd.concat(
    [df, pd.DataFrame([summary])],
    ignore_index=True
)

pd.set_option('display.precision', 3)
df_summary

,Subject,Val size,MC Accuracy,MC F1,MC ITR,MC Acc CI low,MC Acc CI high,SC Accuracy,SC F1,SC ITR,SC Acc CI low,SC Acc CI high
0,S0201,202,0.901,0.444,0.534,0.86,0.942,0.861,0.263,0.419,0.814,0.909
1,S0601,202,0.787,0.189,0.253,0.731,0.844,0.807,0.316,0.292,0.752,0.861
2,S0701,202,0.817,0.213,0.313,0.763,0.87,0.807,0.235,0.292,0.752,0.861
3,S1201,202,0.906,0.24,0.55,0.866,0.946,0.921,0.467,0.601,0.884,0.958
4,S1401,202,0.901,0.412,0.534,0.86,0.942,0.876,0.39,0.46,0.831,0.922
5,S1601,202,0.916,0.452,0.583,0.878,0.954,0.955,0.571,0.737,0.927,0.984
6,S1701,202,0.851,0.211,0.394,0.802,0.901,0.881,0.2,0.474,0.837,0.926
7,S1801,202,0.847,0.205,0.382,0.797,0.896,0.851,0.167,0.394,0.802,0.901
8,S1901,202,0.97,0.769,0.807,0.947,0.994,0.97,0.727,0.807,0.947,0.994
9,S2001,202,0.792,0.222,0.263,0.736,0.848,0.812,0.24,0.302,0.758,0.866


In [ ]:
latex_table = df_summary.to_latex(
    index=False,
    float_format="%.3f",
    caption="Comparison of multi-channel and single-channel EEGNet models",
    label="tab:mc_vs_sc"
)
print(latex_table)

\begin{table}
\caption{Comparison of multi-channel and single-channel EEGNet models}
\label{tab:mc_vs_sc}
\begin{tabular}{lrllllllllll}
\toprule
Subject & Val size & MC Accuracy & MC F1 & MC ITR & MC Acc CI low & MC Acc CI high & SC Accuracy & SC F1 & SC ITR & SC Acc CI low & SC Acc CI high \\
\midrule
S0201 & 202 & 0.901 & 0.444 & 0.534 & 0.860 & 0.942 & 0.861 & 0.263 & 0.419 & 0.814 & 0.909 \\
S0601 & 202 & 0.787 & 0.189 & 0.253 & 0.731 & 0.844 & 0.807 & 0.316 & 0.292 & 0.752 & 0.861 \\
S0701 & 202 & 0.817 & 0.213 & 0.313 & 0.763 & 0.870 & 0.807 & 0.235 & 0.292 & 0.752 & 0.861 \\
S1201 & 202 & 0.906 & 0.240 & 0.550 & 0.866 & 0.946 & 0.921 & 0.467 & 0.601 & 0.884 & 0.958 \\
S1401 & 202 & 0.901 & 0.412 & 0.534 & 0.860 & 0.942 & 0.876 & 0.390 & 0.460 & 0.831 & 0.922 \\
S1601 & 202 & 0.916 & 0.452 & 0.583 & 0.878 & 0.954 & 0.955 & 0.571 & 0.737 & 0.927 & 0.984 \\
S1701 & 202 & 0.851 & 0.211 & 0.394 & 0.802 & 0.901 & 0.881 & 0.200 & 0.474 & 0.837 & 0.926 \\
S1801 & 202 & 0.847 & 0.205 & 0

### BaseCNN, 5 каналов (weight_decay=1e-2)

MC: `BaseCNN(250, 5)` — 5 стэкнутых эпох как каналы.  
SC: `BaseCNN(250, 1)` — 5 эпох усреднены в одну.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs' : 500,
    'lr' : 1e-4,
    'weight_decay' : 1e-2,
    'step_size' : 5,
    'gamma' : 1,
    'num_classes' : 2,
    'model_type' : 'CNN'
}

learning_params_single = {
    'num_epochs' : 500,
    'lr' : 1e-4,
    'weight_decay' : 1e-2,
    'step_size' : 5,
    'gamma' : 1,
    'num_classes' : 2,
    'model_type' : 'CNN'
}

res_basecnn_5ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():

    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = BaseCNN(250, 5).to(my_device)

    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj],
        mc_dataloaders[subj],
        criterion,
        learning_params_mc,
        device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = BaseCNN(250, 1).to(my_device)

    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj],
        single_dataloaders[subj],
        criterion,
        learning_params_single,
        device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_basecnn_5ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (
            acc_mc['Min Accuracy'][-1],
            acc_mc['Max Accuracy'][-1]
        ),
        'sc_ci': (
            acc_sc['Min Accuracy'][-1],
            acc_sc['Max Accuracy'][-1]
        )
    }


===== Subject S1201 =====
Training complete in 0m 54s
Training complete in 0m 49s
MC — Acc: 0.886, ITR: 0.4885, CI: [0.842, 0.930]
SC — Acc: 0.847, ITR: 0.3816, CI: [0.797, 0.896]

===== Subject S0201 =====
Training complete in 0m 54s
Training complete in 0m 52s
MC — Acc: 0.871, ITR: 0.4461, CI: [0.825, 0.917]
SC — Acc: 0.861, ITR: 0.4194, CI: [0.814, 0.909]

===== Subject S0701 =====
Training complete in 0m 54s
Training complete in 0m 51s
MC — Acc: 0.767, ITR: 0.2174, CI: [0.709, 0.826]
SC — Acc: 0.767, ITR: 0.2174, CI: [0.709, 0.826]

===== Subject S0601 =====
Training complete in 0m 54s
Training complete in 0m 51s
MC — Acc: 0.767, ITR: 0.2174, CI: [0.709, 0.826]
SC — Acc: 0.738, ITR: 0.1697, CI: [0.677, 0.798]

===== Subject S1901 =====
Training complete in 0m 54s
Training complete in 0m 50s
MC — Acc: 0.980, ITR: 0.8597, CI: [0.961, 0.999]
SC — Acc: 0.975, ITR: 0.8326, CI: [0.954, 0.997]

===== Subject S2001 =====
Training complete in 0m 52s
Training complete in 0m 49s
MC — Acc: 0.

In [ ]:
import pandas as pd
import numpy as np

rows = []

for subj, res in res_basecnn_5ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],

        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'MC Acc CI low': res['mc_ci'][0],
        'MC Acc CI high': res['mc_ci'][1],

        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
        'SC Acc CI low': res['sc_ci'][0],
        'SC Acc CI high': res['sc_ci'][1],
    })

df = pd.DataFrame(rows).sort_values('Subject')

summary = {
    'Subject': 'Mean ± Std',
    'Val size': df['Val size'].sum(),

    'MC Accuracy': f"{df['MC Accuracy'].mean():.3f} ± {df['MC Accuracy'].std():.3f}",
    'MC F1': f"{df['MC F1'].mean():.3f} ± {df['MC F1'].std():.3f}",
    'MC ITR': f"{df['MC ITR'].mean():.3f} ± {df['MC ITR'].std():.3f}",
    'SC Accuracy': f"{df['SC Accuracy'].mean():.3f} ± {df['SC Accuracy'].std():.3f}",
    'SC F1': f"{df['SC F1'].mean():.3f} ± {df['SC F1'].std():.3f}",
    'SC ITR': f"{df['SC ITR'].mean():.3f} ± {df['SC ITR'].std():.3f}",
    'MC Acc CI low': '',
    'MC Acc CI high': '',
    'SC Acc CI low': '',
    'SC Acc CI high': ''
}

df_summary = pd.concat(
    [df, pd.DataFrame([summary])],
    ignore_index=True
)

pd.set_option('display.precision', 3)
df_summary

,Subject,Val size,MC Accuracy,MC F1,MC ITR,MC Acc CI low,MC Acc CI high,SC Accuracy,SC F1,SC ITR,SC Acc CI low,SC Acc CI high
0,S0201,202,0.871,0.316,0.446,0.825,0.917,0.861,0.263,0.419,0.814,0.909
1,S0601,202,0.767,0.23,0.217,0.709,0.826,0.738,0.232,0.17,0.677,0.798
2,S0701,202,0.767,0.254,0.217,0.709,0.826,0.767,0.254,0.217,0.709,0.826
3,S1201,202,0.886,0.343,0.489,0.842,0.93,0.847,0.244,0.382,0.797,0.896
4,S1401,202,0.896,0.462,0.519,0.854,0.938,0.896,0.462,0.519,0.854,0.938
5,S1601,202,0.931,0.462,0.637,0.896,0.966,0.901,0.412,0.534,0.86,0.942
6,S1701,202,0.861,0.263,0.419,0.814,0.909,0.871,0.235,0.446,0.825,0.917
7,S1801,202,0.822,0.217,0.324,0.769,0.875,0.842,0.273,0.369,0.791,0.892
8,S1901,202,0.98,0.833,0.86,0.961,0.999,0.975,0.762,0.833,0.954,0.997
9,S2001,202,0.752,0.219,0.193,0.693,0.812,0.777,0.211,0.235,0.72,0.835


In [ ]:
latex_table = df_summary.to_latex(
    index=False,
    float_format="%.3f",
    caption="Comparison of multi-channel and single-channel EEGNet models",
    label="tab:mc_vs_sc"
)
print(latex_table)

\begin{table}
\caption{Comparison of multi-channel and single-channel EEGNet models}
\label{tab:mc_vs_sc}
\begin{tabular}{lrllllllllll}
\toprule
Subject & Val size & MC Accuracy & MC F1 & MC ITR & MC Acc CI low & MC Acc CI high & SC Accuracy & SC F1 & SC ITR & SC Acc CI low & SC Acc CI high \\
\midrule
S0201 & 202 & 0.871 & 0.316 & 0.446 & 0.825 & 0.917 & 0.861 & 0.263 & 0.419 & 0.814 & 0.909 \\
S0601 & 202 & 0.767 & 0.230 & 0.217 & 0.709 & 0.826 & 0.738 & 0.232 & 0.170 & 0.677 & 0.798 \\
S0701 & 202 & 0.767 & 0.254 & 0.217 & 0.709 & 0.826 & 0.767 & 0.254 & 0.217 & 0.709 & 0.826 \\
S1201 & 202 & 0.886 & 0.343 & 0.489 & 0.842 & 0.930 & 0.847 & 0.244 & 0.382 & 0.797 & 0.896 \\
S1401 & 202 & 0.896 & 0.462 & 0.519 & 0.854 & 0.938 & 0.896 & 0.462 & 0.519 & 0.854 & 0.938 \\
S1601 & 202 & 0.931 & 0.462 & 0.637 & 0.896 & 0.966 & 0.901 & 0.412 & 0.534 & 0.860 & 0.942 \\
S1701 & 202 & 0.861 & 0.263 & 0.419 & 0.814 & 0.909 & 0.871 & 0.235 & 0.446 & 0.825 & 0.917 \\
S1801 & 202 & 0.822 & 0.217 & 0

## Эксперименты для 10 каналов

In [ ]:
N_CHANNELS = 10

dataloaders = {}

for subj in data.keys():
    train_data, val_data, train_labels, val_labels = train_test_split(
        data[subj],
        labels[subj],
        test_size=0.15,
        shuffle=False
    )

    train_data, train_labels = borderline_smote_torch(
        train_data, train_labels,
        k=15, m=10,
        standardize_for_neighbors=False
    )

    X_train, y_train = build_multichannel_subject_dataset_unique(
        train_data, train_labels, n_channels=N_CHANNELS
    )
    X_val, y_val = build_multichannel_subject_dataset_unique(
        val_data, val_labels, n_channels=N_CHANNELS
    )

    train_dataset = CNNMatrixDataset(
        tensors=(X_train, y_train), with_target=True
    )
    val_dataset = CNNMatrixDataset(
        tensors=(X_val, y_val), with_target=True
    )

    dataloaders[subj] = {
        'train': DataLoader(train_dataset, batch_size=1024, shuffle=True),
        'val': DataLoader(val_dataset, batch_size=1024, shuffle=True),
    }

In [ ]:
mc_dataloaders = {}
single_dataloaders = {}

for subj, dataloader in dataloaders.items():
    # raw датасет уже multi-trial
    X_train, y_train = dataloader['train'].dataset.tensors
    X_val, y_val = dataloader['val'].dataset.tensors

    # single-channel через усреднение
    X_train_single = multichannel_to_single_channel(X_train)
    X_val_single = multichannel_to_single_channel(X_val)

    mc_train_ds = CNNMatrixDataset(
        tensors=(X_train, y_train), with_target=True
    )
    mc_val_ds = CNNMatrixDataset(
        tensors=(X_val, y_val), with_target=True
    )

    single_train_ds = CNNMatrixDataset(
        tensors=(X_train_single, y_train), with_target=True
    )
    single_val_ds = CNNMatrixDataset(
        tensors=(X_val_single, y_val), with_target=True
    )

    mc_dataloaders[subj] = {
        'train': DataLoader(mc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(mc_val_ds, batch_size=256, shuffle=True)
    }

    single_dataloaders[subj] = {
        'train': DataLoader(single_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(single_val_ds, batch_size=256, shuffle=True)
    }

### EEGNet, 10 каналов (weight_decay=1e-5)

MC: `EEGNet(250, 10, F1=128, D=1, F2=256)` — 10 стэкнутых эпох как каналы.  
SC: `EEGNet(250, 1, F1=8, D=8, F2=8)` — 10 эпох усреднены в одну.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_eegnet_10ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():

    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = EEGNet(
        250, 10, F1=128, D=1, F2=256
    ).to(my_device)

    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj],
        mc_dataloaders[subj],
        criterion,
        learning_params_mc,
        device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = EEGNet(
        250, 1, F1=8, D=8, F2=8
    ).to(my_device)

    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj],
        single_dataloaders[subj],
        criterion,
        learning_params_single,
        device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_eegnet_10ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (
            acc_mc['Min Accuracy'][-1],
            acc_mc['Max Accuracy'][-1]
        ),
        'sc_ci': (
            acc_sc['Min Accuracy'][-1],
            acc_sc['Max Accuracy'][-1]
        )
    }


===== Subject S1201 =====
Training complete in 2m 37s
Training complete in 0m 33s
MC — Acc: 0.891, ITR: 0.5034, CI: [0.830, 0.952]
SC — Acc: 0.960, ITR: 0.7595, CI: [0.922, 0.998]

===== Subject S0201 =====
Training complete in 2m 37s
Training complete in 0m 33s
MC — Acc: 0.921, ITR: 0.6006, CI: [0.868, 0.973]
SC — Acc: 0.911, ITR: 0.5665, CI: [0.855, 0.966]

===== Subject S0701 =====
Training complete in 2m 37s
Training complete in 0m 32s
MC — Acc: 0.782, ITR: 0.2438, CI: [0.702, 0.863]
SC — Acc: 0.901, ITR: 0.5342, CI: [0.843, 0.959]

===== Subject S0601 =====
Training complete in 2m 37s
Training complete in 0m 33s
MC — Acc: 0.792, ITR: 0.2625, CI: [0.713, 0.871]
SC — Acc: 0.851, ITR: 0.3939, CI: [0.782, 0.921]

===== Subject S1901 =====
Training complete in 2m 36s
Training complete in 0m 33s
MC — Acc: 1.000, ITR: 1.0000, CI: [1.000, 1.000]
SC — Acc: 1.000, ITR: 1.0000, CI: [1.000, 1.000]

===== Subject S2001 =====
Training complete in 2m 36s
Training complete in 0m 32s
MC — Acc: 0.

In [ ]:
import pandas as pd
import numpy as np

rows = []

for subj, res in res_eegnet_10ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],

        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'MC Acc CI low': res['mc_ci'][0],
        'MC Acc CI high': res['mc_ci'][1],

        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
        'SC Acc CI low': res['sc_ci'][0],
        'SC Acc CI high': res['sc_ci'][1],
    })

df = pd.DataFrame(rows).sort_values('Subject')

summary = {
    'Subject': 'Mean ± Std',
    'Val size': df['Val size'].sum(),

    'MC Accuracy': f"{df['MC Accuracy'].mean():.3f} ± {df['MC Accuracy'].std():.3f}",
    'MC F1': f"{df['MC F1'].mean():.3f} ± {df['MC F1'].std():.3f}",
    'MC ITR': f"{df['MC ITR'].mean():.3f} ± {df['MC ITR'].std():.3f}",
    'SC Accuracy': f"{df['SC Accuracy'].mean():.3f} ± {df['SC Accuracy'].std():.3f}",
    'SC F1': f"{df['SC F1'].mean():.3f} ± {df['SC F1'].std():.3f}",
    'SC ITR': f"{df['SC ITR'].mean():.3f} ± {df['SC ITR'].std():.3f}",
    'MC Acc CI low': '',
    'MC Acc CI high': '',
    'SC Acc CI low': '',
    'SC Acc CI high': ''
}

df_summary = pd.concat(
    [df, pd.DataFrame([summary])],
    ignore_index=True
)

pd.set_option('display.precision', 3)
df_summary

,Subject,Val size,MC Accuracy,MC F1,MC ITR,MC Acc CI low,MC Acc CI high,SC Accuracy,SC F1,SC ITR,SC Acc CI low,SC Acc CI high
0,S0201,101,0.921,0.556,0.601,0.868,0.973,0.911,0.308,0.567,0.855,0.966
1,S0601,101,0.792,0.323,0.263,0.713,0.871,0.851,0.444,0.394,0.782,0.921
2,S0701,101,0.782,0.214,0.244,0.702,0.863,0.901,0.5,0.534,0.843,0.959
3,S1201,101,0.891,0.353,0.503,0.83,0.952,0.96,0.667,0.76,0.922,0.998
4,S1401,100,0.91,0.526,0.564,0.854,0.966,0.93,0.533,0.634,0.88,0.98
5,S1601,101,0.96,0.75,0.76,0.922,0.998,0.98,0.833,0.86,0.953,1.0
6,S1701,101,0.851,0.118,0.394,0.782,0.921,0.911,0.308,0.567,0.855,0.966
7,S1801,101,0.861,0.0,0.419,0.794,0.929,0.891,0.267,0.503,0.83,0.952
8,S1901,101,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,S2001,101,0.832,0.32,0.346,0.759,0.905,0.911,0.4,0.567,0.855,0.966


In [ ]:
latex_table = df_summary.to_latex(
    index=False,
    float_format="%.3f",
    caption="Comparison of multi-channel and single-channel EEGNet models",
    label="tab:mc_vs_sc"
)
print(latex_table)

\begin{table}
\caption{Comparison of multi-channel and single-channel EEGNet models}
\label{tab:mc_vs_sc}
\begin{tabular}{lrllllllllll}
\toprule
Subject & Val size & MC Accuracy & MC F1 & MC ITR & MC Acc CI low & MC Acc CI high & SC Accuracy & SC F1 & SC ITR & SC Acc CI low & SC Acc CI high \\
\midrule
S0201 & 101 & 0.921 & 0.556 & 0.601 & 0.868 & 0.973 & 0.911 & 0.308 & 0.567 & 0.855 & 0.966 \\
S0601 & 101 & 0.792 & 0.323 & 0.263 & 0.713 & 0.871 & 0.851 & 0.444 & 0.394 & 0.782 & 0.921 \\
S0701 & 101 & 0.782 & 0.214 & 0.244 & 0.702 & 0.863 & 0.901 & 0.500 & 0.534 & 0.843 & 0.959 \\
S1201 & 101 & 0.891 & 0.353 & 0.503 & 0.830 & 0.952 & 0.960 & 0.667 & 0.760 & 0.922 & 0.998 \\
S1401 & 100 & 0.910 & 0.526 & 0.564 & 0.854 & 0.966 & 0.930 & 0.533 & 0.634 & 0.880 & 0.980 \\
S1601 & 101 & 0.960 & 0.750 & 0.760 & 0.922 & 0.998 & 0.980 & 0.833 & 0.860 & 0.953 & 1.000 \\
S1701 & 101 & 0.851 & 0.118 & 0.394 & 0.782 & 0.921 & 0.911 & 0.308 & 0.567 & 0.855 & 0.966 \\
S1801 & 101 & 0.861 & 0.000 & 0

### BaseCNN, 10 каналов (weight_decay=1e-2)

MC: `BaseCNN(250, 10)` — 10 стэкнутых эпох как каналы.  
SC: `BaseCNN(250, 1)` — 10 эпох усреднены в одну.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs' : 500,
    'lr' : 1e-4,
    'weight_decay' : 1e-2,
    'step_size' : 5,
    'gamma' : 1,
    'num_classes' : 2,
    'model_type' : 'CNN'
}

learning_params_single = {
    'num_epochs' : 500,
    'lr' : 1e-4,
    'weight_decay' : 1e-2,
    'step_size' : 5,
    'gamma' : 1,
    'num_classes' : 2,
    'model_type' : 'CNN'
}

res_basecnn_10ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():

    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = BaseCNN(250, 10).to(my_device)

    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj],
        mc_dataloaders[subj],
        criterion,
        learning_params_mc,
        device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = BaseCNN(250, 1).to(my_device)

    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj],
        single_dataloaders[subj],
        criterion,
        learning_params_single,
        device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_basecnn_10ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (
            acc_mc['Min Accuracy'][-1],
            acc_mc['Max Accuracy'][-1]
        ),
        'sc_ci': (
            acc_sc['Min Accuracy'][-1],
            acc_sc['Max Accuracy'][-1]
        )
    }


===== Subject S1201 =====
Training complete in 0m 31s
Training complete in 0m 27s
MC — Acc: 0.950, ITR: 0.7157, CI: [0.908, 0.993]
SC — Acc: 0.941, ITR: 0.6749, CI: [0.894, 0.987]

===== Subject S0201 =====
Training complete in 0m 29s
Training complete in 0m 27s
MC — Acc: 0.851, ITR: 0.3939, CI: [0.782, 0.921]
SC — Acc: 0.851, ITR: 0.3939, CI: [0.782, 0.921]

===== Subject S0701 =====
Training complete in 0m 29s
Training complete in 0m 26s
MC — Acc: 0.881, ITR: 0.4741, CI: [0.818, 0.944]
SC — Acc: 0.812, ITR: 0.3025, CI: [0.736, 0.888]

===== Subject S0601 =====
Training complete in 0m 30s
Training complete in 0m 27s
MC — Acc: 0.832, ITR: 0.3462, CI: [0.759, 0.905]
SC — Acc: 0.822, ITR: 0.3238, CI: [0.747, 0.896]

===== Subject S1901 =====
Training complete in 0m 29s
Training complete in 0m 26s
MC — Acc: 0.990, ITR: 0.9199, CI: [0.971, 1.000]
SC — Acc: 1.000, ITR: 1.0000, CI: [1.000, 1.000]

===== Subject S2001 =====
Training complete in 0m 29s
Training complete in 0m 27s
MC — Acc: 0.

In [ ]:
import pandas as pd
import numpy as np

rows = []

for subj, res in res_basecnn_10ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],

        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'MC Acc CI low': res['mc_ci'][0],
        'MC Acc CI high': res['mc_ci'][1],

        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
        'SC Acc CI low': res['sc_ci'][0],
        'SC Acc CI high': res['sc_ci'][1],
    })

df = pd.DataFrame(rows).sort_values('Subject')

summary = {
    'Subject': 'Mean ± Std',
    'Val size': df['Val size'].sum(),

    'MC Accuracy': f"{df['MC Accuracy'].mean():.3f} ± {df['MC Accuracy'].std():.3f}",
    'MC F1': f"{df['MC F1'].mean():.3f} ± {df['MC F1'].std():.3f}",
    'MC ITR': f"{df['MC ITR'].mean():.3f} ± {df['MC ITR'].std():.3f}",
    'SC Accuracy': f"{df['SC Accuracy'].mean():.3f} ± {df['SC Accuracy'].std():.3f}",
    'SC F1': f"{df['SC F1'].mean():.3f} ± {df['SC F1'].std():.3f}",
    'SC ITR': f"{df['SC ITR'].mean():.3f} ± {df['SC ITR'].std():.3f}",
    'MC Acc CI low': '',
    'MC Acc CI high': '',
    'SC Acc CI low': '',
    'SC Acc CI high': ''
}

df_summary = pd.concat(
    [df, pd.DataFrame([summary])],
    ignore_index=True
)

pd.set_option('display.precision', 3)
df_summary

,Subject,Val size,MC Accuracy,MC F1,MC ITR,MC Acc CI low,MC Acc CI high,SC Accuracy,SC F1,SC ITR,SC Acc CI low,SC Acc CI high
0,S0201,101,0.851,0.211,0.394,0.782,0.921,0.851,0.348,0.394,0.782,0.921
1,S0601,101,0.832,0.37,0.346,0.759,0.905,0.822,0.4,0.324,0.747,0.896
2,S0701,101,0.881,0.455,0.474,0.818,0.944,0.812,0.174,0.302,0.736,0.888
3,S1201,101,0.95,0.615,0.716,0.908,0.993,0.941,0.5,0.675,0.894,0.987
4,S1401,100,0.92,0.5,0.598,0.867,0.973,0.93,0.533,0.634,0.88,0.98
5,S1601,101,0.931,0.533,0.637,0.881,0.98,0.97,0.8,0.807,0.937,1.0
6,S1701,101,0.842,0.273,0.369,0.77,0.913,0.861,0.222,0.419,0.794,0.929
7,S1801,101,0.871,0.235,0.446,0.806,0.937,0.832,0.105,0.346,0.759,0.905
8,S1901,101,0.99,0.909,0.92,0.971,1.0,1.0,1.0,1.0,1.0,1.0
9,S2001,101,0.842,0.2,0.369,0.77,0.913,0.881,0.333,0.474,0.818,0.944


In [ ]:
latex_table = df_summary.to_latex(
    index=False,
    float_format="%.3f",
    caption="Comparison of multi-channel and single-channel EEGNet models",
    label="tab:mc_vs_sc"
)
print(latex_table)

\begin{table}
\caption{Comparison of multi-channel and single-channel EEGNet models}
\label{tab:mc_vs_sc}
\begin{tabular}{lrllllllllll}
\toprule
Subject & Val size & MC Accuracy & MC F1 & MC ITR & MC Acc CI low & MC Acc CI high & SC Accuracy & SC F1 & SC ITR & SC Acc CI low & SC Acc CI high \\
\midrule
S0201 & 101 & 0.851 & 0.211 & 0.394 & 0.782 & 0.921 & 0.851 & 0.348 & 0.394 & 0.782 & 0.921 \\
S0601 & 101 & 0.832 & 0.370 & 0.346 & 0.759 & 0.905 & 0.822 & 0.400 & 0.324 & 0.747 & 0.896 \\
S0701 & 101 & 0.881 & 0.455 & 0.474 & 0.818 & 0.944 & 0.812 & 0.174 & 0.302 & 0.736 & 0.888 \\
S1201 & 101 & 0.950 & 0.615 & 0.716 & 0.908 & 0.993 & 0.941 & 0.500 & 0.675 & 0.894 & 0.987 \\
S1401 & 100 & 0.920 & 0.500 & 0.598 & 0.867 & 0.973 & 0.930 & 0.533 & 0.634 & 0.880 & 0.980 \\
S1601 & 101 & 0.931 & 0.533 & 0.637 & 0.881 & 0.980 & 0.970 & 0.800 & 0.807 & 0.937 & 1.000 \\
S1701 & 101 & 0.842 & 0.273 & 0.369 & 0.770 & 0.913 & 0.861 & 0.222 & 0.419 & 0.794 & 0.929 \\
S1801 & 101 & 0.871 & 0.235 & 0

## SVM с усреднением эпох

Группируем N эпох → усредняем → обучаем SVM на усреднённых данных.


In [ ]:
from sklearn.svm import SVC

def compute_itr(accuracy, n_classes=2):
    """
    Wolpaw's Information Transfer Rate (bits/trial).
    ITR = log2(N) + P*log2(P) + (1-P)*log2((1-P)/(N-1))
    """
    P = float(accuracy)
    N = int(n_classes)
    if P <= 0 or P >= 1:
        P = np.clip(P, 1e-10, 1 - 1e-10)
    if N <= 1:
        return 0.0
    return float(np.log2(N) + P * np.log2(P) + (1 - P) * np.log2((1 - P) / (N - 1)))

def get_metrics(preds, labels, n_classes=2):
    corrects = (preds == labels).sum()
    min_acc, max_acc = proportion_confint(corrects, len(preds), 0.05)
    accuracy = corrects / len(preds)
    itr = compute_itr(accuracy, n_classes)
    return {'Accuracy': accuracy, 'ITR': itr, 'Min Accuracy': min_acc, 'Max Accuracy': max_acc}


### Общая модель SVM (усреднение N эпох)


In [ ]:
res_svm_epoch_common = {}

for N_CHANNELS in [5, 10]:
    all_train_data = []
    all_train_labels = []
    all_val_data = []
    all_val_labels = []

    for subj in data.keys():
        train_data, val_data, train_labels, val_labels = train_test_split(
            data[subj],
            labels[subj],
            test_size=0.15,
            shuffle=False
        )

        train_data, train_labels = borderline_smote_torch(
            train_data, train_labels,
            k=15, m=10,
            standardize_for_neighbors=False
        )

        # Группируем эпохи в наборы по N_CHANNELS и усредняем
        X_train, y_train = build_multichannel_subject_dataset_unique(
            train_data, train_labels, n_channels=N_CHANNELS
        )
        X_val, y_val = build_multichannel_subject_dataset_unique(
            val_data, val_labels, n_channels=N_CHANNELS
        )

        # Усредняем N эпох → (T', features)
        X_train_avg = X_train.mean(dim=1).numpy()
        y_train_avg = y_train.numpy()
        X_val_avg = X_val.mean(dim=1).numpy()
        y_val_avg = y_val.numpy()

        all_train_data.append(X_train_avg)
        all_train_labels.append(y_train_avg)
        all_val_data.append(X_val_avg)
        all_val_labels.append(y_val_avg)

    all_train_data = np.vstack(all_train_data)
    all_train_labels = np.hstack(all_train_labels)
    all_val_data = np.vstack(all_val_data)
    all_val_labels = np.hstack(all_val_labels)

    clf = SVC().fit(all_train_data, all_train_labels)
    preds = clf.predict(all_val_data)

    common_acc = get_metrics(preds, all_val_labels)
    res_svm_epoch_common[N_CHANNELS] = {
        'accuracy': common_acc['Accuracy'],
        'itr': common_acc['ITR'],
    }
    print(f"=== N_CHANNELS = {N_CHANNELS} ===")
    print(f"Common Accuracy: {common_acc['Accuracy']:.3f}")
    print(f"Common ITR: {common_acc['ITR']:.4f} bits/trial")
    print(f"Common accuracy CI: [{round(common_acc['Min Accuracy'], 3)}, {round(common_acc['Max Accuracy'], 3)}]")
    print()


=== N_CHANNELS = 5 ===
Common Accuracy: 0.840
Common ITR: 0.3647 bits/trial
Common accuracy CI: [0.824, 0.856]

=== N_CHANNELS = 10 ===
Common Accuracy: 0.891
Common ITR: 0.5031 bits/trial
Common accuracy CI: [0.872, 0.91]



### Индивидуальные модели SVM (усреднение N эпох)


In [ ]:
res_svm_epoch = {}

for N_CHANNELS in [5, 10]:
    print(f"=== N_CHANNELS = {N_CHANNELS} ===")

    for subj in data.keys():
        train_data, val_data, train_labels, val_labels = train_test_split(
            data[subj],
            labels[subj],
            test_size=0.15,
            shuffle=False
        )

        train_data, train_labels = borderline_smote_torch(
            train_data, train_labels,
            k=15, m=10,
            standardize_for_neighbors=False
        )

        X_train, y_train = build_multichannel_subject_dataset_unique(
            train_data, train_labels, n_channels=N_CHANNELS
        )
        X_val, y_val = build_multichannel_subject_dataset_unique(
            val_data, val_labels, n_channels=N_CHANNELS
        )

        X_train_avg = X_train.mean(dim=1).numpy()
        y_train_avg = y_train.numpy()
        X_val_avg = X_val.mean(dim=1).numpy()
        y_val_avg = y_val.numpy()

        clf = SVC().fit(X_train_avg, y_train_avg)
        preds = clf.predict(X_val_avg)

        acc = get_metrics(preds, y_val_avg)
        print(f"{subj} — Accuracy: {acc['Accuracy']:.3f}, ITR: {acc['ITR']:.4f} bits/trial, "
              f"CI: [{acc['Min Accuracy']:.3f}, {acc['Max Accuracy']:.3f}]")

        res_svm_epoch[(N_CHANNELS, subj)] = {
            'accuracy': acc['Accuracy'],
            'itr': acc['ITR'],
            'size': len(y_val_avg),
            'lower_ci': acc['Min Accuracy'],
            'upper_ci': acc['Max Accuracy']
        }
    print()


=== N_CHANNELS = 5 ===
S1201 — Accuracy: 0.926, ITR: 0.6184 bits/trial, CI: [0.890, 0.962]
S0201 — Accuracy: 0.881, ITR: 0.4741 bits/trial, CI: [0.837, 0.926]
S0701 — Accuracy: 0.842, ITR: 0.3695 bits/trial, CI: [0.791, 0.892]
S0601 — Accuracy: 0.792, ITR: 0.2625 bits/trial, CI: [0.736, 0.848]
S1901 — Accuracy: 0.975, ITR: 0.8326 bits/trial, CI: [0.954, 0.997]
S2001 — Accuracy: 0.797, ITR: 0.2722 bits/trial, CI: [0.742, 0.852]
S1701 — Accuracy: 0.881, ITR: 0.4741 bits/trial, CI: [0.837, 0.926]
S1401 — Accuracy: 0.896, ITR: 0.5186 bits/trial, CI: [0.854, 0.938]
S1801 — Accuracy: 0.856, ITR: 0.4065 bits/trial, CI: [0.808, 0.905]
S1601 — Accuracy: 0.946, ITR: 0.6950 bits/trial, CI: [0.914, 0.977]

=== N_CHANNELS = 10 ===
S1201 — Accuracy: 0.941, ITR: 0.6749 bits/trial, CI: [0.894, 0.987]
S0201 — Accuracy: 0.931, ITR: 0.6367 bits/trial, CI: [0.881, 0.980]
S0701 — Accuracy: 0.941, ITR: 0.6749 bits/trial, CI: [0.894, 0.987]
S0601 — Accuracy: 0.891, ITR: 0.5034 bits/trial, CI: [0.830, 0.952]


In [ ]:
import pandas as pd
import numpy as np

for N_CH in [5, 10]:
    rows = []
    for (n, subj), res in res_svm_epoch.items():
        if n != N_CH:
            continue
        rows.append({
            'Subject': subj,
            'Val size': res['size'],
            'Accuracy': res['accuracy'],
            'ITR': res['itr'],
            'CI low': res['lower_ci'],
            'CI high': res['upper_ci'],
        })

    df = pd.DataFrame(rows).sort_values('Subject')

    summary = {
        'Subject': 'Mean ± Std',
        'Val size': df['Val size'].sum(),
        'Accuracy': f"{df['Accuracy'].mean():.3f} ± {df['Accuracy'].std():.3f}",
        'ITR': f"{df['ITR'].mean():.3f} ± {df['ITR'].std():.3f}",
        'CI low': '',
        'CI high': '',
    }

    df_summary = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
    print(f'\n=== SVM Individual, N_CHANNELS = {N_CH} ===')
    display(df_summary)


=== SVM Individual, N_CHANNELS = 5 ===


,Subject,Val size,Accuracy,ITR,CI low,CI high
0,S0201,202,0.881,0.474,0.837,0.926
1,S0601,202,0.792,0.263,0.736,0.848
2,S0701,202,0.842,0.369,0.791,0.892
3,S1201,202,0.926,0.618,0.89,0.962
4,S1401,202,0.896,0.519,0.854,0.938
5,S1601,202,0.946,0.695,0.914,0.977
6,S1701,202,0.881,0.474,0.837,0.926
7,S1801,202,0.856,0.407,0.808,0.905
8,S1901,202,0.975,0.833,0.954,0.997
9,S2001,202,0.797,0.272,0.742,0.852



=== SVM Individual, N_CHANNELS = 10 ===


,Subject,Val size,Accuracy,ITR,CI low,CI high
0,S0201,101,0.931,0.637,0.881,0.98
1,S0601,101,0.891,0.503,0.83,0.952
2,S0701,101,0.941,0.675,0.894,0.987
3,S1201,101,0.941,0.675,0.894,0.987
4,S1401,100,0.97,0.806,0.937,1.0
5,S1601,101,0.96,0.76,0.922,0.998
6,S1701,101,0.921,0.601,0.868,0.973
7,S1801,101,0.941,0.675,0.894,0.987
8,S1901,101,1.0,1.0,1.0,1.0
9,S2001,101,0.861,0.419,0.794,0.929


## Усреднение разных пользователей (кросс-субъектное)

Вместо усреднения N эпох одного субъекта, стэкуем/усредняем один trial от каждого субъекта.  
Каждый канал = trial одного пользователя (всего 10 субъектов → 10 каналов).

- **MC**: CNN видит матрицу `(n_subjects, features)` — все субъекты как каналы.  
- **SC**: Усреднение по субъектам → `(1, features)` — один средний trial.


In [ ]:
# --- 1. Находим минимальное число сэмплов каждого класса (после split + SMOTE) ---
min_pos_train, min_neg_train = float('inf'), float('inf')
min_pos_val, min_neg_val = float('inf'), float('inf')

split_cache = {}
for subj in data.keys():
    train_data_s, val_data_s, train_labels_s, val_labels_s = train_test_split(
        data[subj], labels[subj], test_size=0.15, shuffle=False
    )
    train_data_s, train_labels_s = borderline_smote_torch(
        train_data_s, train_labels_s, k=15, m=10, standardize_for_neighbors=False
    )
    split_cache[subj] = (train_data_s, val_data_s, train_labels_s, val_labels_s)

    min_pos_train = min(min_pos_train, int((train_labels_s == 1).sum()))
    min_neg_train = min(min_neg_train, int((train_labels_s == 0).sum()))
    min_pos_val = min(min_pos_val, int((val_labels_s == 1).sum()))
    min_neg_val = min(min_neg_val, int((val_labels_s == 0).sum()))

# --- 2. Собираем каналы: один канал = один субъект ---
channels_pos_train, channels_neg_train = [], []
channels_pos_val, channels_neg_val = [], []

for subj in data.keys():
    tr_data, vl_data, tr_labels, vl_labels = split_cache[subj]

    channels_pos_train.append(
        tr_data[tr_labels == 1][torch.randperm(int((tr_labels == 1).sum()))][:min_pos_train].unsqueeze(1)
    )
    channels_neg_train.append(
        tr_data[tr_labels == 0][torch.randperm(int((tr_labels == 0).sum()))][:min_neg_train].unsqueeze(1)
    )
    channels_pos_val.append(
        vl_data[vl_labels == 1][torch.randperm(int((vl_labels == 1).sum()))][:min_pos_val].unsqueeze(1)
    )
    channels_neg_val.append(
        vl_data[vl_labels == 0][torch.randperm(int((vl_labels == 0).sum()))][:min_neg_val].unsqueeze(1)
    )

# --- 3. Конкатенация: (n_samples, n_subjects, features) ---
N_SUBJECTS = len(data.keys())

cs_train_data = torch.cat([
    torch.cat(channels_pos_train, dim=1),
    torch.cat(channels_neg_train, dim=1)
], dim=0)

cs_val_data = torch.cat([
    torch.cat(channels_pos_val, dim=1),
    torch.cat(channels_neg_val, dim=1)
], dim=0)

cs_train_labels = torch.cat([
    torch.ones(min_pos_train, dtype=torch.float64),
    torch.zeros(min_neg_train, dtype=torch.float64)
])

cs_val_labels = torch.cat([
    torch.ones(min_pos_val, dtype=torch.float64),
    torch.zeros(min_neg_val, dtype=torch.float64)
])

print(f'Субъектов: {N_SUBJECTS}')
print(f'Train: {cs_train_data.shape}, Val: {cs_val_data.shape}')
print(f'Train labels: pos={min_pos_train}, neg={min_neg_train}')
print(f'Val labels:   pos={min_pos_val}, neg={min_neg_val}')


Субъектов: 10
Train: torch.Size([10758, 10, 250]), Val: torch.Size([997, 10, 250])
Train labels: pos=5379, neg=5379
Val labels:   pos=54, neg=943


In [ ]:
# MC-датасет: (n_samples, n_subjects, features)
cs_mc_train_ds = CNNMatrixDataset(tensors=(cs_train_data, cs_train_labels), with_target=True)
cs_mc_val_ds = CNNMatrixDataset(tensors=(cs_val_data, cs_val_labels), with_target=True)

# SC-датасет: усреднение по субъектам → (n_samples, 1, features)
cs_sc_train_ds = CNNMatrixDataset(
    tensors=(multichannel_to_single_channel(cs_train_data), cs_train_labels), with_target=True
)
cs_sc_val_ds = CNNMatrixDataset(
    tensors=(multichannel_to_single_channel(cs_val_data), cs_val_labels), with_target=True
)

cs_mc_dataloader = {
    'train': DataLoader(cs_mc_train_ds, batch_size=256, shuffle=True),
    'val': DataLoader(cs_mc_val_ds, batch_size=256, shuffle=True)
}
cs_sc_dataloader = {
    'train': DataLoader(cs_sc_train_ds, batch_size=256, shuffle=True),
    'val': DataLoader(cs_sc_val_ds, batch_size=256, shuffle=True)
}


### EEGNet, кросс-субъектное усреднение (weight_decay=1e-5)

MC: `EEGNet(250, N_SUBJECTS, F1=128, D=1, F2=256)` — каждый субъект как канал.  
SC: `EEGNet(250, 1, F1=8, D=8, F2=8)` — усреднение по субъектам.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

# ===== MULTI-CHANNEL =====
cs_mc_model = EEGNet(250, N_SUBJECTS, F1=128, D=1, F2=256).to(my_device)
loss_mc, acc_mc, _ = train_model(
    cs_mc_model, cs_mc_dataloader, criterion, learning_params_mc, device=my_device
)

# ===== SINGLE-CHANNEL (MEAN) =====
cs_sc_model = EEGNet(250, 1, F1=8, D=8, F2=8).to(my_device)
loss_sc, acc_sc, _ = train_model(
    cs_sc_model, cs_sc_dataloader, criterion, learning_params_single, device=my_device
)

print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

res_cs_eegnet = {
    'mc_accuracy': acc_mc['Accuracy'][-1],
    'mc_itr': acc_mc['ITR'][-1],
    'sc_accuracy': acc_sc['Accuracy'][-1],
    'sc_itr': acc_sc['ITR'][-1],
}


Training complete in 25m 48s
Training complete in 5m 2s
MC — Acc: 0.970, ITR: 0.8052, CI: [0.959, 0.981]
SC — Acc: 0.887, ITR: 0.4901, CI: [0.867, 0.906]


### BaseCNN, кросс-субъектное усреднение (weight_decay=1e-2)

MC: `BaseCNN(250, N_SUBJECTS)` — каждый субъект как канал.  
SC: `BaseCNN(250, 1)` — усреднение по субъектам.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

# ===== MULTI-CHANNEL =====
cs_mc_model = BaseCNN(250, N_SUBJECTS).to(my_device)
loss_mc, acc_mc, _ = train_model(
    cs_mc_model, cs_mc_dataloader, criterion, learning_params_mc, device=my_device
)

# ===== SINGLE-CHANNEL (MEAN) =====
cs_sc_model = BaseCNN(250, 1).to(my_device)
loss_sc, acc_sc, _ = train_model(
    cs_sc_model, cs_sc_dataloader, criterion, learning_params_single, device=my_device
)

print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

res_cs_basecnn = {
    'mc_accuracy': acc_mc['Accuracy'][-1],
    'mc_itr': acc_mc['ITR'][-1],
    'sc_accuracy': acc_sc['Accuracy'][-1],
    'sc_itr': acc_sc['ITR'][-1],
}


Training complete in 4m 23s
Training complete in 4m 4s
MC — Acc: 0.915, ITR: 0.5796, CI: [0.897, 0.932]
SC — Acc: 0.873, ITR: 0.4498, CI: [0.852, 0.893]


### SVM, кросс-субъектное усреднение

Усредняем trials всех субъектов → `(n_samples, features)` → SVM.


In [ ]:
# Усредняем по субъектам: (n_samples, n_subjects, features) → (n_samples, features)
cs_train_avg = cs_train_data.mean(dim=1).numpy()
cs_val_avg = cs_val_data.mean(dim=1).numpy()
cs_train_lbl = cs_train_labels.numpy()
cs_val_lbl = cs_val_labels.numpy()

clf = SVC().fit(cs_train_avg, cs_train_lbl)
preds = clf.predict(cs_val_avg)

cs_svm_acc = get_metrics(preds, cs_val_lbl)
print(f'SVM (кросс-субъект) — Acc: {cs_svm_acc["Accuracy"]:.3f}, ITR: {cs_svm_acc["ITR"]:.4f} bits/trial, '
      f'CI: [{cs_svm_acc["Min Accuracy"]:.3f}, {cs_svm_acc["Max Accuracy"]:.3f}]')

res_cs_svm = {
    'accuracy': cs_svm_acc['Accuracy'],
    'itr': cs_svm_acc['ITR'],
}


SVM (кросс-субъект) — Acc: 0.885, ITR: 0.4842 bits/trial, CI: [0.865, 0.904]


## Смешанное усреднение (разные пользователи × разные trials)

Объединяем все trials всех субъектов в один пул, затем группируем N случайных trials в каналы.  
Каждый канал может содержать trial любого субъекта — смешение подходов 2 (усреднение эпох) и 3 (кросс-субъектное).

- **MC**: CNN видит `(N, features)` — N случайных trials из общего пула.  
- **SC**: Усреднение N trials → `(1, features)`.


In [ ]:
# --- Пул: объединяем все субъекты в один набор (после split + SMOTE) ---
pool_train_data = []
pool_train_labels = []
pool_val_data = []
pool_val_labels = []

for subj in data.keys():
    train_data_s, val_data_s, train_labels_s, val_labels_s = train_test_split(
        data[subj], labels[subj], test_size=0.15, shuffle=False
    )
    train_data_s, train_labels_s = borderline_smote_torch(
        train_data_s, train_labels_s, k=15, m=10, standardize_for_neighbors=False
    )
    pool_train_data.append(train_data_s)
    pool_train_labels.append(train_labels_s)
    pool_val_data.append(val_data_s)
    pool_val_labels.append(val_labels_s)

pool_train_data = torch.cat(pool_train_data, dim=0)
pool_train_labels = torch.cat(pool_train_labels, dim=0)
pool_val_data = torch.cat(pool_val_data, dim=0)
pool_val_labels = torch.cat(pool_val_labels, dim=0)

print(f'Pool train: {pool_train_data.shape}, labels: {pool_train_labels.shape}')
print(f'Pool val:   {pool_val_data.shape}, labels: {pool_val_labels.shape}')


Pool train: torch.Size([107718, 250]), labels: torch.Size([107718])
Pool val:   torch.Size([10140, 250]), labels: torch.Size([10140])


In [ ]:
# Группируем случайные trials из пула в каналы (N=5 и N=10)
mix_mc_dataloaders = {}
mix_sc_dataloaders = {}

for N_CH in [5, 10]:
    X_train, y_train = build_multichannel_subject_dataset_unique(
        pool_train_data, pool_train_labels, n_channels=N_CH
    )
    X_val, y_val = build_multichannel_subject_dataset_unique(
        pool_val_data, pool_val_labels, n_channels=N_CH
    )

    X_train_single = multichannel_to_single_channel(X_train)
    X_val_single = multichannel_to_single_channel(X_val)

    mc_train_ds = CNNMatrixDataset(tensors=(X_train, y_train), with_target=True)
    mc_val_ds = CNNMatrixDataset(tensors=(X_val, y_val), with_target=True)
    sc_train_ds = CNNMatrixDataset(tensors=(X_train_single, y_train), with_target=True)
    sc_val_ds = CNNMatrixDataset(tensors=(X_val_single, y_val), with_target=True)

    mix_mc_dataloaders[N_CH] = {
        'train': DataLoader(mc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(mc_val_ds, batch_size=256, shuffle=True)
    }
    mix_sc_dataloaders[N_CH] = {
        'train': DataLoader(sc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(sc_val_ds, batch_size=256, shuffle=True)
    }

    print(f'N={N_CH}: MC train={X_train.shape}, SC train={X_train_single.shape}')


N=5: MC train=torch.Size([21542, 5, 250]), SC train=torch.Size([21542, 1, 250])
N=10: MC train=torch.Size([10770, 10, 250]), SC train=torch.Size([10770, 1, 250])


### EEGNet, смешанное усреднение (weight_decay=1e-5)

MC: `EEGNet(250, N, ...)` — N случайных trials из пула всех субъектов.  
SC: `EEGNet(250, 1, ...)` — те же N trials усреднены в один.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_mix_eegnet = {}

for N_CH in [5, 10]:
    print(f'\n===== N = {N_CH} =====')

    # ===== MULTI-CHANNEL =====
    mix_mc_model = EEGNet(250, N_CH, F1=128, D=1, F2=256).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        mix_mc_model, mix_mc_dataloaders[N_CH], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    mix_sc_model = EEGNet(250, 1, F1=8, D=8, F2=8).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        mix_sc_model, mix_sc_dataloaders[N_CH], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_mix_eegnet[N_CH] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_itr': acc_sc['ITR'][-1],
    }



===== N = 5 =====
Training complete in 32m 23s
Training complete in 10m 3s
MC — Acc: 0.763, ITR: 0.2103, CI: [0.745, 0.782]
SC — Acc: 0.827, ITR: 0.3362, CI: [0.811, 0.844]

===== N = 10 =====
Training complete in 25m 49s
Training complete in 5m 13s
MC — Acc: 0.858, ITR: 0.4102, CI: [0.836, 0.879]
SC — Acc: 0.884, ITR: 0.4808, CI: [0.864, 0.903]


### BaseCNN, смешанное усреднение (weight_decay=1e-2)

MC: `BaseCNN(250, N)` — N случайных trials из пула всех субъектов.  
SC: `BaseCNN(250, 1)` — те же N trials усреднены в один.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_mix_basecnn = {}

for N_CH in [5, 10]:
    print(f'\n===== N = {N_CH} =====')

    # ===== MULTI-CHANNEL =====
    mix_mc_model = BaseCNN(250, N_CH).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        mix_mc_model, mix_mc_dataloaders[N_CH], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    mix_sc_model = BaseCNN(250, 1).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        mix_sc_model, mix_sc_dataloaders[N_CH], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_mix_basecnn[N_CH] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_itr': acc_sc['ITR'][-1],
    }



===== N = 5 =====
Training complete in 8m 18s
Training complete in 7m 56s
MC — Acc: 0.797, ITR: 0.2726, CI: [0.780, 0.815]
SC — Acc: 0.808, ITR: 0.2946, CI: [0.791, 0.825]

===== N = 10 =====
Training complete in 4m 21s
Training complete in 3m 59s
MC — Acc: 0.874, ITR: 0.4526, CI: [0.853, 0.894]
SC — Acc: 0.886, ITR: 0.4895, CI: [0.867, 0.906]


### SVM, смешанное усреднение

Группируем N случайных trials из пула → усредняем → SVM.


In [ ]:
res_mix_svm = {}

for N_CH in [5, 10]:
    X_train, y_train = build_multichannel_subject_dataset_unique(
        pool_train_data, pool_train_labels, n_channels=N_CH
    )
    X_val, y_val = build_multichannel_subject_dataset_unique(
        pool_val_data, pool_val_labels, n_channels=N_CH
    )

    # Усредняем N trials → (T', features)
    X_train_avg = X_train.mean(dim=1).numpy()
    y_train_avg = y_train.numpy()
    X_val_avg = X_val.mean(dim=1).numpy()
    y_val_avg = y_val.numpy()

    clf = SVC().fit(X_train_avg, y_train_avg)
    preds = clf.predict(X_val_avg)

    acc = get_metrics(preds, y_val_avg)
    print(f'N={N_CH} — Acc: {acc["Accuracy"]:.3f}, ITR: {acc["ITR"]:.4f} bits/trial, '
          f'CI: [{acc["Min Accuracy"]:.3f}, {acc["Max Accuracy"]:.3f}]')

    res_mix_svm[N_CH] = {
        'accuracy': acc['Accuracy'],
        'itr': acc['ITR'],
    }


N=5 — Acc: 0.822, ITR: 0.3241 bits/trial, CI: [0.805, 0.839]
N=10 — Acc: 0.880, ITR: 0.4694 bits/trial, CI: [0.860, 0.900]


## Кросс-субъектное усреднение с несколькими trials (K trials × все субъекты)

Берём **всех** субъектов, но от каждого — по **K trials** (K=2, 3).  
Итого каналов = N_subjects × K (например, 10×2=20 или 10×3=30).

Каждый сэмпл гарантированно содержит trials **от всех** субъектов — контролируемое смешение.


In [ ]:
# --- Split + SMOTE для каждого субъекта (кэшируем) ---
strat_cache = {}
for subj in data.keys():
    tr_d, vl_d, tr_l, vl_l = train_test_split(
        data[subj], labels[subj], test_size=0.15, shuffle=False
    )
    tr_d, tr_l = borderline_smote_torch(
        tr_d, tr_l, k=15, m=10, standardize_for_neighbors=False
    )
    strat_cache[subj] = (tr_d, vl_d, tr_l, vl_l)

subj_list = list(strat_cache.keys())
N_SUBJECTS = len(subj_list)
print(f'Субъектов: {N_SUBJECTS}')

# --- Строим датасет: K trials от каждого субъекта ---
strat_mc_dataloaders = {}
strat_sc_dataloaders = {}

for K in [2, 3]:
    n_channels = N_SUBJECTS * K

    X_all, y_all = {}, {}
    for phase in ['train', 'val']:
        X_cls_list = []
        y_cls_list = []
        for cls in [1, 0]:
            # min trials этого класса у самого бедного субъекта
            if phase == 'train':
                min_trials = min(int((strat_cache[s][2] == cls).sum()) for s in subj_list)
            else:
                min_trials = min(int((strat_cache[s][3] == cls).sum()) for s in subj_list)

            n_groups = min_trials // K
            if n_groups == 0:
                raise ValueError(f'K={K}, class={cls}: недостаточно trials (min={min_trials})')

            channels = []
            for s in subj_list:
                if phase == 'train':
                    d, l = strat_cache[s][0], strat_cache[s][2]
                else:
                    d, l = strat_cache[s][1], strat_cache[s][3]

                d_cls = d[l == cls]
                perm = torch.randperm(len(d_cls))[:n_groups * K]
                # (n_groups, K, features)
                d_cls_grouped = d_cls[perm].view(n_groups, K, *d_cls.shape[1:])
                channels.append(d_cls_grouped)

            # channels: list of (n_groups, K, features) per subject
            # stack → (n_groups, N_SUBJECTS, K, features) → reshape → (n_groups, N_SUBJECTS*K, features)
            stacked = torch.stack(channels, dim=1)  # (n_groups, N_SUBJECTS, K, features)
            stacked = stacked.view(n_groups, n_channels, *d_cls.shape[1:])

            X_cls_list.append(stacked)
            y_cls_list.append(torch.full((n_groups,), cls, dtype=torch.float64))

        X_all[phase] = torch.cat(X_cls_list, dim=0)
        y_all[phase] = torch.cat(y_cls_list, dim=0)

    # SC: усреднение
    X_train_sc = multichannel_to_single_channel(X_all['train'])
    X_val_sc = multichannel_to_single_channel(X_all['val'])

    mc_train_ds = CNNMatrixDataset(tensors=(X_all['train'], y_all['train']), with_target=True)
    mc_val_ds = CNNMatrixDataset(tensors=(X_all['val'], y_all['val']), with_target=True)
    sc_train_ds = CNNMatrixDataset(tensors=(X_train_sc, y_all['train']), with_target=True)
    sc_val_ds = CNNMatrixDataset(tensors=(X_val_sc, y_all['val']), with_target=True)

    strat_mc_dataloaders[K] = {
        'train': DataLoader(mc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(mc_val_ds, batch_size=256, shuffle=True)
    }
    strat_sc_dataloaders[K] = {
        'train': DataLoader(sc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(sc_val_ds, batch_size=256, shuffle=True)
    }

    print(f'K={K}: каналов={n_channels}, MC train={X_all["train"].shape}, SC train={X_train_sc.shape}')


Субъектов: 10
K=2: каналов=20, MC train=torch.Size([5378, 20, 250]), SC train=torch.Size([5378, 1, 250])
K=3: каналов=30, MC train=torch.Size([3586, 30, 250]), SC train=torch.Size([3586, 1, 250])


### EEGNet, K trials × все субъекты (weight_decay=1e-5)

MC: `EEGNet(250, N_subj*K, ...)` — K trials от каждого субъекта.  
SC: `EEGNet(250, 1, ...)` — те же каналы усреднены в один.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_strat_eegnet = {}

for K in [2, 3]:
    n_ch = N_SUBJECTS * K
    print(f'\n===== K = {K}  (каналов = {n_ch}) =====')

    # ===== MULTI-CHANNEL =====
    strat_eeg_mc = EEGNet(250, n_ch, F1=128, D=1, F2=256).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        strat_eeg_mc, strat_mc_dataloaders[K], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    strat_eeg_sc = EEGNet(250, 1, F1=8, D=8, F2=8).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        strat_eeg_sc, strat_sc_dataloaders[K], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_strat_eegnet[K] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_itr': acc_sc['ITR'][-1],
    }



===== K = 2  (каналов = 20) =====
Training complete in 21m 56s
Training complete in 2m 33s
MC — Acc: 0.978, ITR: 0.8470, CI: [0.965, 0.991]
SC — Acc: 0.960, ITR: 0.7570, CI: [0.943, 0.977]

===== K = 3  (каналов = 30) =====
Training complete in 23m 4s
Training complete in 1m 46s
MC — Acc: 0.985, ITR: 0.8873, CI: [0.972, 0.998]
SC — Acc: 0.976, ITR: 0.8361, CI: [0.959, 0.992]


### BaseCNN, K trials × все субъекты (weight_decay=1e-2)

MC: `BaseCNN(250, N_subj*K)` — K trials от каждого субъекта.  
SC: `BaseCNN(250, 1)` — те же каналы усреднены в один.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_strat_basecnn = {}

for K in [2, 3]:
    n_ch = N_SUBJECTS * K
    print(f'\n===== K = {K}  (каналов = {n_ch}) =====')

    # ===== MULTI-CHANNEL =====
    strat_cnn_mc = BaseCNN(250, n_ch).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        strat_cnn_mc, strat_mc_dataloaders[K], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    strat_cnn_sc = BaseCNN(250, 1).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        strat_cnn_sc, strat_sc_dataloaders[K], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_strat_basecnn[K] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_itr': acc_sc['ITR'][-1],
    }



===== K = 2  (каналов = 20) =====
Training complete in 2m 34s
Training complete in 2m 1s
MC — Acc: 0.972, ITR: 0.8152, CI: [0.957, 0.986]
SC — Acc: 0.954, ITR: 0.7300, CI: [0.935, 0.972]

===== K = 3  (каналов = 30) =====
Training complete in 1m 47s
Training complete in 1m 22s
MC — Acc: 0.970, ITR: 0.8050, CI: [0.951, 0.988]
SC — Acc: 0.961, ITR: 0.7616, CI: [0.940, 0.982]


### SVM, K trials × все субъекты

Усредняем N_subj×K каналов → 1 вектор → SVM.


In [ ]:
res_strat_svm = {}

for K in [2, 3]:
    n_ch = N_SUBJECTS * K

    # MC данные уже готовы в strat_mc_dataloaders; для SVM берём из X_all напрямую
    # Пересобираем, т.к. X_all перезаписывался в цикле — пересоздаём
    X_all_svm, y_all_svm = {}, {}
    for phase in ['train', 'val']:
        X_cls_list = []
        y_cls_list = []
        for cls in [1, 0]:
            if phase == 'train':
                min_trials = min(int((strat_cache[s][2] == cls).sum()) for s in subj_list)
            else:
                min_trials = min(int((strat_cache[s][3] == cls).sum()) for s in subj_list)
            n_groups = min_trials // K

            channels = []
            for s in subj_list:
                if phase == 'train':
                    d, l = strat_cache[s][0], strat_cache[s][2]
                else:
                    d, l = strat_cache[s][1], strat_cache[s][3]
                d_cls = d[l == cls]
                perm = torch.randperm(len(d_cls))[:n_groups * K]
                d_cls_grouped = d_cls[perm].view(n_groups, K, *d_cls.shape[1:])
                channels.append(d_cls_grouped)

            stacked = torch.stack(channels, dim=1).view(n_groups, n_ch, *d_cls.shape[1:])
            X_cls_list.append(stacked)
            y_cls_list.append(torch.full((n_groups,), cls, dtype=torch.float64))

        X_all_svm[phase] = torch.cat(X_cls_list, dim=0)
        y_all_svm[phase] = torch.cat(y_cls_list, dim=0)

    # Усредняем все каналы → плоский вектор
    X_train_avg = X_all_svm['train'].mean(dim=1).numpy()
    y_train_avg = y_all_svm['train'].numpy()
    X_val_avg = X_all_svm['val'].mean(dim=1).numpy()
    y_val_avg = y_all_svm['val'].numpy()

    clf = SVC().fit(X_train_avg, y_train_avg)
    preds = clf.predict(X_val_avg)

    acc = get_metrics(preds, y_val_avg)
    print(f'K={K} (каналов={n_ch}) — Acc: {acc["Accuracy"]:.3f}, ITR: {acc["ITR"]:.4f} bits/trial, '
          f'CI: [{acc["Min Accuracy"]:.3f}, {acc["Max Accuracy"]:.3f}]')

    res_strat_svm[K] = {
        'accuracy': acc['Accuracy'],
        'itr': acc['ITR'],
    }


K=2 (каналов=20) — Acc: 0.930, ITR: 0.6330 bits/trial, CI: [0.907, 0.952]
K=3 (каналов=30) — Acc: 0.967, ITR: 0.7901 bits/trial, CI: [0.948, 0.986]


## Сдвиг во времени (Time-Shifted Channels)

Используем длинные эпохи (500 отсчётов = 2 с при 250 Гц) из `Samara_data`.
Из каждой эпохи вырезаем **N каналов** длиной 250 отсчётов (1 с), сдвинутых на 100 мс:

| Канал | Начало | Конец | Интервал (мс) |
|-------|--------|-------|----------------|
| Ch 0  |   0    |  250  |   0–1000       |
| Ch 1  |  25    |  275  | 100–1100       |
| Ch 2  |  50    |  300  | 200–1200       |
| …     |  …     |  …    |      …         |
| Ch 10 | 250    |  500  | 1000–2000      |

Максимум 11 каналов при shift=100 мс (дальше выходим за границу 2-секундной эпохи).

- **MC**: CNN видит `(N, 250)` — N временных окон одного trial.
- **SC**: Усреднение N окон → `(1, 250)` — один средний сигнал.


In [ ]:
from time_shift import load_p300_subjects, build_timeshifted_dataset, time_shift_info

# --- Загрузка 500-точечных эпох из Samara_data ---
SAMARA_PATH = '/content/drive/MyDrive/pattern_recognition/Samara_data/'
ts_raw_data, ts_raw_labels = load_p300_subjects(SAMARA_PATH)

print(f'Loaded {len(ts_raw_data)} subjects from Samara_data')
for subj in sorted(ts_raw_data.keys()):
    n_pos = int((ts_raw_labels[subj] == 1).sum())
    print(f'  {subj}: {ts_raw_data[subj].shape}, pos={n_pos}')

# --- Параметры time-shift ---
SHIFT_MS = 100
N_CHANNELS_LIST = [5]

for n_ch in N_CHANNELS_LIST:
    info = time_shift_info(500, 250, 250, SHIFT_MS, n_ch)
    print(f'\nN={n_ch}: shift={SHIFT_MS}ms ({info["shift_samples"]} samples), '
          f'last end={info["last_channel_end_sample"]} ({info["last_channel_end_ms"]:.0f}ms), '
          f'max possible={info["max_channels"]}')

# --- Split + SMOTE + time-shift → MC/SC dataloaders per subject ---
ts_mc_dataloaders = {}
ts_sc_dataloaders = {}

for N_CH in N_CHANNELS_LIST:
    ts_mc_dataloaders[N_CH] = {}
    ts_sc_dataloaders[N_CH] = {}

    for subj in sorted(ts_raw_data.keys()):
        X_raw = torch.from_numpy(ts_raw_data[subj]).float()
        y_raw = torch.from_numpy(ts_raw_labels[subj]).long()

        train_data, val_data, train_labels, val_labels = train_test_split(
            X_raw, y_raw, test_size=0.15, shuffle=False
        )

        train_data, train_labels = borderline_smote_torch(
            train_data, train_labels, k=15, m=10, standardize_for_neighbors=False
        )

        # MC: (n_samples, N_CH, 250)
        X_train_mc, y_tr = build_timeshifted_dataset(
            train_data.numpy(), train_labels.numpy(),
            shift_ms=SHIFT_MS, n_channels=N_CH
        )
        X_val_mc, y_vl = build_timeshifted_dataset(
            val_data.numpy(), val_labels.numpy(),
            shift_ms=SHIFT_MS, n_channels=N_CH
        )

        # SC: average → (n_samples, 1, 250)
        X_train_sc = X_train_mc.mean(axis=1, keepdims=True)
        X_val_sc = X_val_mc.mean(axis=1, keepdims=True)

        # Torch tensors
        X_tr_mc_t = torch.from_numpy(X_train_mc).float()
        y_tr_t = torch.from_numpy(y_tr).long()
        X_vl_mc_t = torch.from_numpy(X_val_mc).float()
        y_vl_t = torch.from_numpy(y_vl).long()

        X_tr_sc_t = torch.from_numpy(X_train_sc).float()
        X_vl_sc_t = torch.from_numpy(X_val_sc).float()

        mc_tr_ds = CNNMatrixDataset(tensors=(X_tr_mc_t, y_tr_t), with_target=True)
        mc_vl_ds = CNNMatrixDataset(tensors=(X_vl_mc_t, y_vl_t), with_target=True)
        sc_tr_ds = CNNMatrixDataset(tensors=(X_tr_sc_t, y_tr_t), with_target=True)
        sc_vl_ds = CNNMatrixDataset(tensors=(X_vl_sc_t, y_vl_t), with_target=True)

        ts_mc_dataloaders[N_CH][subj] = {
            'train': DataLoader(mc_tr_ds, batch_size=256, shuffle=True),
            'val': DataLoader(mc_vl_ds, batch_size=256, shuffle=True)
        }
        ts_sc_dataloaders[N_CH][subj] = {
            'train': DataLoader(sc_tr_ds, batch_size=256, shuffle=True),
            'val': DataLoader(sc_vl_ds, batch_size=256, shuffle=True)
        }

    print(f'\nN_CH={N_CH}: dataloaders ready for {len(ts_mc_dataloaders[N_CH])} subjects, '
          f'train MC={X_tr_mc_t.shape}, SC={X_tr_sc_t.shape}')


Loaded 10 subjects from Samara_data
  S0201: (6769, 500), pos=420
  S0601: (6768, 500), pos=426
  S0701: (6760, 500), pos=422
  S1201: (6760, 500), pos=428
  S1401: (6760, 500), pos=424
  S1601: (6760, 500), pos=421
  S1701: (6760, 500), pos=422
  S1801: (6761, 500), pos=420
  S1901: (6759, 500), pos=415
  S2001: (6760, 500), pos=417

N=5: shift=100ms (25 samples), last end=350 (1400ms), max possible=11

N_CH=5: dataloaders ready for 10 subjects, train MC=torch.Size([10780, 5, 250]), SC=torch.Size([10780, 1, 250])


### EEGNet, time-shifted channels (weight_decay=1e-5)

MC: `EEGNet(250, N, F1=128, D=1, F2=256)` — N временных окон (сдвиг 100 мс) одного trial.
SC: `EEGNet(250, 1, F1=8, D=8, F2=8)` — те же N окон усреднены в одно.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_ts_eegnet = {}

for N_CH in N_CHANNELS_LIST:
    print(f'\n{"="*60}')
    print(f'  EEGNet — N_CHANNELS = {N_CH}, shift = {SHIFT_MS}ms')
    print(f'{"="*60}')

    for subj in sorted(ts_mc_dataloaders[N_CH].keys()):
        print(f'\n--- Subject {subj} ---')

        # ===== MULTI-CHANNEL =====
        mc_model = EEGNet(250, N_CH, F1=128, D=1, F2=256).to(my_device)
        loss_mc, acc_mc, _ = train_model(
            mc_model, ts_mc_dataloaders[N_CH][subj],
            criterion, learning_params_mc, device=my_device
        )

        # ===== SINGLE-CHANNEL (MEAN) =====
        sc_model = EEGNet(250, 1, F1=8, D=8, F2=8).to(my_device)
        loss_sc, acc_sc, _ = train_model(
            sc_model, ts_sc_dataloaders[N_CH][subj],
            criterion, learning_params_single, device=my_device
        )

        print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
        print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

        res_ts_eegnet[(N_CH, subj)] = {
            'mc_accuracy': acc_mc['Accuracy'][-1],
            'mc_itr': acc_mc['ITR'][-1],
            'sc_accuracy': acc_sc['Accuracy'][-1],
            'sc_itr': acc_sc['ITR'][-1],
            'size': len(ts_mc_dataloaders[N_CH][subj]['val'].dataset),
            'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
            'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
        }



  EEGNet — N_CHANNELS = 5, shift = 100ms

--- Subject S0201 ---
Training complete in 16m 39s
Training complete in 5m 6s
MC — Acc: 0.837, ITR: 0.3577, CI: [0.814, 0.859]
SC — Acc: 0.737, ITR: 0.1691, CI: [0.710, 0.764]

--- Subject S0601 ---
Training complete in 16m 27s
Training complete in 4m 59s
MC — Acc: 0.850, ITR: 0.3911, CI: [0.828, 0.872]
SC — Acc: 0.743, ITR: 0.1780, CI: [0.716, 0.770]

--- Subject S0701 ---
Training complete in 16m 27s
Training complete in 4m 48s
MC — Acc: 0.810, ITR: 0.2978, CI: [0.786, 0.834]
SC — Acc: 0.720, ITR: 0.1444, CI: [0.692, 0.748]

--- Subject S1201 ---
Training complete in 16m 23s
Training complete in 4m 47s
MC — Acc: 0.818, ITR: 0.3146, CI: [0.794, 0.841]
SC — Acc: 0.717, ITR: 0.1404, CI: [0.689, 0.745]

--- Subject S1401 ---
Training complete in 16m 12s
Training complete in 4m 45s
MC — Acc: 0.837, ITR: 0.3592, CI: [0.815, 0.860]
SC — Acc: 0.716, ITR: 0.1391, CI: [0.688, 0.744]

--- Subject S1601 ---
Training complete in 16m 22s
Training complete

In [ ]:
import pandas as pd
import numpy as np

for N_CH in N_CHANNELS_LIST:
    rows = []
    for (n, subj), res in res_ts_eegnet.items():
        if n != N_CH:
            continue
        rows.append({
            'Subject': subj,
            'Val size': res['size'],
            'MC Accuracy': res['mc_accuracy'],
            'MC ITR': res['mc_itr'],
            'MC CI low': res['mc_ci'][0],
            'MC CI high': res['mc_ci'][1],
            'SC Accuracy': res['sc_accuracy'],
            'SC ITR': res['sc_itr'],
            'SC CI low': res['sc_ci'][0],
            'SC CI high': res['sc_ci'][1],
        })

    df = pd.DataFrame(rows).sort_values('Subject')
    summary = {
        'Subject': 'Mean ± Std',
        'Val size': df['Val size'].sum(),
        'MC Accuracy': f"{df['MC Accuracy'].mean():.3f} ± {df['MC Accuracy'].std():.3f}",
        'MC ITR': f"{df['MC ITR'].mean():.3f} ± {df['MC ITR'].std():.3f}",
        'SC Accuracy': f"{df['SC Accuracy'].mean():.3f} ± {df['SC Accuracy'].std():.3f}",
        'SC ITR': f"{df['SC ITR'].mean():.3f} ± {df['SC ITR'].std():.3f}",
        'MC CI low': '', 'MC CI high': '', 'SC CI low': '', 'SC CI high': '',
    }
    df_summary = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
    print(f'\n=== EEGNet Time-Shifted, N_CHANNELS = {N_CH} ===')
    display(df_summary)

### BaseCNN, time-shifted channels (weight_decay=1e-2)

MC: `BaseCNN(250, N)` — N временных окон (сдвиг 100 мс) одного trial.
SC: `BaseCNN(250, 1)` — те же N окон усреднены в одно.


In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_ts_basecnn = {}

for N_CH in N_CHANNELS_LIST:
    print(f'\n{"="*60}')
    print(f'  BaseCNN — N_CHANNELS = {N_CH}, shift = {SHIFT_MS}ms')
    print(f'{"="*60}')

    for subj in sorted(ts_mc_dataloaders[N_CH].keys()):
        print(f'\n--- Subject {subj} ---')

        # ===== MULTI-CHANNEL =====
        mc_model = BaseCNN(250, N_CH).to(my_device)
        loss_mc, acc_mc, _ = train_model(
            mc_model, ts_mc_dataloaders[N_CH][subj],
            criterion, learning_params_mc, device=my_device
        )

        # ===== SINGLE-CHANNEL (MEAN) =====
        sc_model = BaseCNN(250, 1).to(my_device)
        loss_sc, acc_sc, _ = train_model(
            sc_model, ts_sc_dataloaders[N_CH][subj],
            criterion, learning_params_single, device=my_device
        )

        print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
        print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

        res_ts_basecnn[(N_CH, subj)] = {
            'mc_accuracy': acc_mc['Accuracy'][-1],
            'mc_itr': acc_mc['ITR'][-1],
            'sc_accuracy': acc_sc['Accuracy'][-1],
            'sc_itr': acc_sc['ITR'][-1],
            'size': len(ts_mc_dataloaders[N_CH][subj]['val'].dataset),
            'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
            'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
        }



  BaseCNN — N_CHANNELS = 5, shift = 100ms

--- Subject S0201 ---
Training complete in 5m 40s
Training complete in 4m 15s
MC — Acc: 0.692, ITR: 0.1091, CI: [0.664, 0.720]
SC — Acc: 0.732, ITR: 0.1618, CI: [0.705, 0.760]

--- Subject S0601 ---
Training complete in 5m 34s
Training complete in 4m 16s
MC — Acc: 0.678, ITR: 0.0936, CI: [0.649, 0.707]
SC — Acc: 0.680, ITR: 0.0957, CI: [0.651, 0.709]

--- Subject S0701 ---
Training complete in 5m 25s
Training complete in 4m 13s
MC — Acc: 0.655, ITR: 0.0703, CI: [0.626, 0.684]
SC — Acc: 0.663, ITR: 0.0778, CI: [0.634, 0.692]

--- Subject S1201 ---
Training complete in 5m 21s
Training complete in 4m 13s
MC — Acc: 0.709, ITR: 0.1301, CI: [0.681, 0.737]
SC — Acc: 0.661, ITR: 0.0759, CI: [0.632, 0.690]

--- Subject S1401 ---
Training complete in 5m 19s
Training complete in 4m 15s
MC — Acc: 0.734, ITR: 0.1639, CI: [0.707, 0.761]
SC — Acc: 0.686, ITR: 0.1027, CI: [0.658, 0.715]

--- Subject S1601 ---
Training complete in 5m 16s
Training complete in 

In [ ]:
import pandas as pd
import numpy as np

for N_CH in N_CHANNELS_LIST:
    rows = []
    for (n, subj), res in res_ts_basecnn.items():
        if n != N_CH:
            continue
        rows.append({
            'Subject': subj,
            'Val size': res['size'],
            'MC Accuracy': res['mc_accuracy'],
            'MC ITR': res['mc_itr'],
            'MC CI low': res['mc_ci'][0],
            'MC CI high': res['mc_ci'][1],
            'SC Accuracy': res['sc_accuracy'],
            'SC ITR': res['sc_itr'],
            'SC CI low': res['sc_ci'][0],
            'SC CI high': res['sc_ci'][1],
        })

    df = pd.DataFrame(rows).sort_values('Subject')
    summary = {
        'Subject': 'Mean ± Std',
        'Val size': df['Val size'].sum(),
        'MC Accuracy': f"{df['MC Accuracy'].mean():.3f} ± {df['MC Accuracy'].std():.3f}",
        'MC ITR': f"{df['MC ITR'].mean():.3f} ± {df['MC ITR'].std():.3f}",
        'SC Accuracy': f"{df['SC Accuracy'].mean():.3f} ± {df['SC Accuracy'].std():.3f}",
        'SC ITR': f"{df['SC ITR'].mean():.3f} ± {df['SC ITR'].std():.3f}",
        'MC CI low': '', 'MC CI high': '', 'SC CI low': '', 'SC CI high': '',
    }
    df_summary = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
    print(f'\n=== BaseCNN Time-Shifted, N_CHANNELS = {N_CH} ===')
    display(df_summary)


=== BaseCNN Time-Shifted, N_CHANNELS = 5 ===


,Subject,Val size,MC Accuracy,MC ITR,MC CI low,MC CI high,SC Accuracy,SC ITR,SC CI low,SC CI high
0,S0201,1016,0.691929,0.109066,0.66354,0.720319,0.732283,0.161831,0.705058,0.759509
1,S0601,1016,0.67815,0.093618,0.649423,0.706877,0.680118,0.095747,0.651438,0.708799
2,S0701,1014,0.654832,0.070322,0.62557,0.684095,0.662722,0.07781,0.633622,0.691822
3,S1201,1014,0.709073,0.130084,0.681118,0.737028,0.66075,0.0759,0.631608,0.689891
4,S1401,1014,0.733728,0.163936,0.706522,0.760933,0.686391,0.102704,0.657834,0.714947
5,S1601,1014,0.781065,0.241782,0.755613,0.806518,0.707101,0.127563,0.67909,0.735112
6,S1701,1014,0.759369,0.203912,0.733058,0.785679,0.658777,0.074016,0.629595,0.687959
7,S1801,1015,0.646305,0.062676,0.616892,0.675719,0.580296,0.018684,0.549935,0.610656
8,S1901,1014,0.921105,0.601725,0.904512,0.937697,0.831361,0.345418,0.808315,0.854407
9,S2001,1014,0.708087,0.12882,0.680103,0.73607,0.676529,0.091883,0.647735,0.705322


### SVM, time-shifted channels

Усредняем N временных окон → 1 вектор (250 признаков) → SVM.
Общая модель (все субъекты вместе) и индивидуальные модели.


In [ ]:
from sklearn.svm import SVC

res_ts_svm_common = {}
res_ts_svm_indiv = {}

for N_CH in N_CHANNELS_LIST:
    print(f'\n{"="*60}')
    print(f'  SVM — N_CHANNELS = {N_CH}, shift = {SHIFT_MS}ms')
    print(f'{"="*60}')

    # --- Общая модель: пулим все субъекты ---
    all_train_data, all_train_labels = [], []
    all_val_data, all_val_labels = [], []

    for subj in sorted(ts_mc_dataloaders[N_CH].keys()):
        X_tr, y_tr = ts_mc_dataloaders[N_CH][subj]['train'].dataset.tensors
        X_vl, y_vl = ts_mc_dataloaders[N_CH][subj]['val'].dataset.tensors

        # Усредняем каналы → (n_samples, 250)
        all_train_data.append(X_tr.mean(dim=1).numpy())
        all_train_labels.append(y_tr.numpy())
        all_val_data.append(X_vl.mean(dim=1).numpy())
        all_val_labels.append(y_vl.numpy())

    all_train_data = np.vstack(all_train_data)
    all_train_labels = np.hstack(all_train_labels)
    all_val_data = np.vstack(all_val_data)
    all_val_labels = np.hstack(all_val_labels)

    clf = SVC().fit(all_train_data, all_train_labels)
    preds = clf.predict(all_val_data)
    acc = get_metrics(preds, all_val_labels)
    res_ts_svm_common[N_CH] = {
        'accuracy': acc['Accuracy'],
        'itr': acc['ITR'],
    }
    print(f'\nОбщая SVM — Acc: {acc["Accuracy"]:.3f}, ITR: {acc["ITR"]:.4f} bits/trial, '
          f'CI: [{acc["Min Accuracy"]:.3f}, {acc["Max Accuracy"]:.3f}]')

    # --- Индивидуальные модели ---
    print(f'\nИндивидуальные модели:')
    for subj in sorted(ts_mc_dataloaders[N_CH].keys()):
        X_tr, y_tr = ts_mc_dataloaders[N_CH][subj]['train'].dataset.tensors
        X_vl, y_vl = ts_mc_dataloaders[N_CH][subj]['val'].dataset.tensors

        X_tr_avg = X_tr.mean(dim=1).numpy()
        y_tr_np = y_tr.numpy()
        X_vl_avg = X_vl.mean(dim=1).numpy()
        y_vl_np = y_vl.numpy()

        clf_ind = SVC().fit(X_tr_avg, y_tr_np)
        preds_ind = clf_ind.predict(X_vl_avg)
        acc_ind = get_metrics(preds_ind, y_vl_np)
        print(f'  {subj} — Acc: {acc_ind["Accuracy"]:.3f}, ITR: {acc_ind["ITR"]:.4f}, '
              f'CI: [{acc_ind["Min Accuracy"]:.3f}, {acc_ind["Max Accuracy"]:.3f}]')

        res_ts_svm_indiv[(N_CH, subj)] = {
            'accuracy': acc_ind['Accuracy'],
            'itr': acc_ind['ITR'],
            'size': len(y_vl_np),
            'lower_ci': acc_ind['Min Accuracy'],
            'upper_ci': acc_ind['Max Accuracy'],
        }



  SVM — N_CHANNELS = 5, shift = 100ms

Общая SVM — Acc: 0.760, ITR: 0.2049 bits/trial, CI: [0.752, 0.768]

Индивидуальные модели:
  S0201 — Acc: 0.857, ITR: 0.4087, CI: [0.836, 0.879]
  S0601 — Acc: 0.840, ITR: 0.3647, CI: [0.817, 0.862]
  S0701 — Acc: 0.839, ITR: 0.3639, CI: [0.817, 0.862]
  S1201 — Acc: 0.822, ITR: 0.3254, CI: [0.799, 0.846]
  S1401 — Acc: 0.770, ITR: 0.2224, CI: [0.744, 0.796]
  S1601 — Acc: 0.829, ITR: 0.3409, CI: [0.806, 0.853]
  S1701 — Acc: 0.865, ITR: 0.4287, CI: [0.844, 0.886]
  S1801 — Acc: 0.764, ITR: 0.2109, CI: [0.737, 0.790]
  S1901 — Acc: 0.930, ITR: 0.6340, CI: [0.914, 0.946]
  S2001 — Acc: 0.818, ITR: 0.3146, CI: [0.794, 0.841]


In [ ]:
import pandas as pd
import numpy as np

for N_CH in N_CHANNELS_LIST:
    rows = []
    for (n, subj), res in res_ts_svm_indiv.items():
        if n != N_CH:
            continue
        rows.append({
            'Subject': subj,
            'Val size': res['size'],
            'Accuracy': res['accuracy'],
            'ITR': res['itr'],
            'CI low': res['lower_ci'],
            'CI high': res['upper_ci'],
        })

    df = pd.DataFrame(rows).sort_values('Subject')
    summary = {
        'Subject': 'Mean ± Std',
        'Val size': df['Val size'].sum(),
        'Accuracy': f"{df['Accuracy'].mean():.3f} ± {df['Accuracy'].std():.3f}",
        'ITR': f"{df['ITR'].mean():.3f} ± {df['ITR'].std():.3f}",
        'CI low': '', 'CI high': '',
    }
    df_summary = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
    common = res_ts_svm_common[N_CH]
    print(f'\n=== SVM Time-Shifted, N_CHANNELS = {N_CH} ===')
    print(f'Common SVM — Acc: {common["accuracy"]:.3f}, ITR: {common["itr"]:.4f}')
    print(f'Individual SVMs:')
    display(df_summary)


=== SVM Time-Shifted, N_CHANNELS = 5 ===
Common SVM — Acc: 0.760, ITR: 0.2049
Individual SVMs:


,Subject,Val size,Accuracy,ITR,CI low,CI high
0,S0201,1016,0.857283,0.408691,0.835775,0.878791
1,S0601,1016,0.839567,0.364655,0.817,0.862134
2,S0701,1014,0.83925,0.3639,0.816643,0.861858
3,S1201,1014,0.822485,0.32539,0.798967,0.846004
4,S1401,1014,0.770217,0.222367,0.744323,0.796111
5,S1601,1014,0.829389,0.340898,0.806235,0.852542
6,S1701,1014,0.864892,0.428716,0.843851,0.885932
7,S1801,1015,0.763547,0.210908,0.737407,0.789687
8,S1901,1014,0.92998,0.634003,0.914274,0.945687
9,S2001,1014,0.817554,0.314602,0.793783,0.841326


## Unified Comparison Across All Approaches

Summary table comparing all data aggregation strategies.

- **Per-subject experiments** (Epoch Avg, Time-Shifted): Mean ± Std across subjects
- **Global experiments** (Cross-Subject, Mixed, K-Trials): single model metric

Columns: EEGNet MC/SC, BaseCNN MC/SC, SVM (averaged or common)

In [ ]:
import pandas as pd
import numpy as np

# --- Helper: extract mean from per-subject dict (keyed by (N_CH, subj) or subj) ---
def _mean_mc_sc(res_dict, n_ch=None):
    """Extract mean MC/SC accuracy & ITR from a per-subject result dict."""
    vals = {}
    for key, res in res_dict.items():
        if n_ch is not None:
            if isinstance(key, tuple) and key[0] != n_ch:
                continue
        for metric in ['mc_accuracy', 'mc_itr', 'sc_accuracy', 'sc_itr']:
            vals.setdefault(metric, []).append(res[metric])
    return {k: np.mean(v) for k, v in vals.items()}

def _mean_svm(res_dict, n_ch=None):
    """Extract mean accuracy & ITR from a per-subject SVM result dict."""
    vals = {}
    for key, res in res_dict.items():
        if n_ch is not None:
            if isinstance(key, tuple) and key[0] != n_ch:
                continue
        for metric in ['accuracy', 'itr']:
            vals.setdefault(metric, []).append(res[metric])
    return {k: np.mean(v) for k, v in vals.items()}

def _fmt(val):
    return f'{val:.3f}' if isinstance(val, (float, np.floating)) else str(val)

# --- Build rows ---
rows = []

# 1) Epoch averaging N=5
ea5_eeg = _mean_mc_sc(res_eegnet_5ch)
ea5_cnn = _mean_mc_sc(res_basecnn_5ch)
ea5_svm = _mean_svm(res_svm_epoch, n_ch=5)
rows.append({
    'Approach': 'Epoch Avg', 'Config': 'N=5',
    'EEGNet MC Acc': ea5_eeg['mc_accuracy'], 'EEGNet MC ITR': ea5_eeg['mc_itr'],
    'EEGNet SC Acc': ea5_eeg['sc_accuracy'], 'EEGNet SC ITR': ea5_eeg['sc_itr'],
    'BaseCNN MC Acc': ea5_cnn['mc_accuracy'], 'BaseCNN MC ITR': ea5_cnn['mc_itr'],
    'BaseCNN SC Acc': ea5_cnn['sc_accuracy'], 'BaseCNN SC ITR': ea5_cnn['sc_itr'],
    'SVM Acc': ea5_svm['accuracy'], 'SVM ITR': ea5_svm['itr'],
})

# 2) Epoch averaging N=10
ea10_eeg = _mean_mc_sc(res_eegnet_10ch)
ea10_cnn = _mean_mc_sc(res_basecnn_10ch)
ea10_svm = _mean_svm(res_svm_epoch, n_ch=10)
rows.append({
    'Approach': 'Epoch Avg', 'Config': 'N=10',
    'EEGNet MC Acc': ea10_eeg['mc_accuracy'], 'EEGNet MC ITR': ea10_eeg['mc_itr'],
    'EEGNet SC Acc': ea10_eeg['sc_accuracy'], 'EEGNet SC ITR': ea10_eeg['sc_itr'],
    'BaseCNN MC Acc': ea10_cnn['mc_accuracy'], 'BaseCNN MC ITR': ea10_cnn['mc_itr'],
    'BaseCNN SC Acc': ea10_cnn['sc_accuracy'], 'BaseCNN SC ITR': ea10_cnn['sc_itr'],
    'SVM Acc': ea10_svm['accuracy'], 'SVM ITR': ea10_svm['itr'],
})

# 3) Cross-subject (N=10 subjects)
rows.append({
    'Approach': 'Cross-Subject', 'Config': 'N=10',
    'EEGNet MC Acc': res_cs_eegnet['mc_accuracy'], 'EEGNet MC ITR': res_cs_eegnet['mc_itr'],
    'EEGNet SC Acc': res_cs_eegnet['sc_accuracy'], 'EEGNet SC ITR': res_cs_eegnet['sc_itr'],
    'BaseCNN MC Acc': res_cs_basecnn['mc_accuracy'], 'BaseCNN MC ITR': res_cs_basecnn['mc_itr'],
    'BaseCNN SC Acc': res_cs_basecnn['sc_accuracy'], 'BaseCNN SC ITR': res_cs_basecnn['sc_itr'],
    'SVM Acc': res_cs_svm['accuracy'], 'SVM ITR': res_cs_svm['itr'],
})

# 4) Mixed averaging
for N_CH in [5, 10]:
    rows.append({
        'Approach': 'Mixed', 'Config': f'N={N_CH}',
        'EEGNet MC Acc': res_mix_eegnet[N_CH]['mc_accuracy'], 'EEGNet MC ITR': res_mix_eegnet[N_CH]['mc_itr'],
        'EEGNet SC Acc': res_mix_eegnet[N_CH]['sc_accuracy'], 'EEGNet SC ITR': res_mix_eegnet[N_CH]['sc_itr'],
        'BaseCNN MC Acc': res_mix_basecnn[N_CH]['mc_accuracy'], 'BaseCNN MC ITR': res_mix_basecnn[N_CH]['mc_itr'],
        'BaseCNN SC Acc': res_mix_basecnn[N_CH]['sc_accuracy'], 'BaseCNN SC ITR': res_mix_basecnn[N_CH]['sc_itr'],
        'SVM Acc': res_mix_svm[N_CH]['accuracy'], 'SVM ITR': res_mix_svm[N_CH]['itr'],
    })

# 5) K trials x all subjects
for K in [2, 3]:
    n_ch = 10 * K  # N_SUBJECTS * K
    rows.append({
        'Approach': 'K Trials x Subj', 'Config': f'K={K} (N={n_ch})',
        'EEGNet MC Acc': res_strat_eegnet[K]['mc_accuracy'], 'EEGNet MC ITR': res_strat_eegnet[K]['mc_itr'],
        'EEGNet SC Acc': res_strat_eegnet[K]['sc_accuracy'], 'EEGNet SC ITR': res_strat_eegnet[K]['sc_itr'],
        'BaseCNN MC Acc': res_strat_basecnn[K]['mc_accuracy'], 'BaseCNN MC ITR': res_strat_basecnn[K]['mc_itr'],
        'BaseCNN SC Acc': res_strat_basecnn[K]['sc_accuracy'], 'BaseCNN SC ITR': res_strat_basecnn[K]['sc_itr'],
        'SVM Acc': res_strat_svm[K]['accuracy'], 'SVM ITR': res_strat_svm[K]['itr'],
    })

# 6) Time-shifted
for N_CH in [5, 11]:
    ts_eeg = _mean_mc_sc(res_ts_eegnet, n_ch=N_CH)
    ts_cnn = _mean_mc_sc(res_ts_basecnn, n_ch=N_CH)
    ts_svm = _mean_svm(res_ts_svm_indiv, n_ch=N_CH)
    rows.append({
        'Approach': 'Time-Shifted', 'Config': f'N={N_CH}',
        'EEGNet MC Acc': ts_eeg['mc_accuracy'], 'EEGNet MC ITR': ts_eeg['mc_itr'],
        'EEGNet SC Acc': ts_eeg['sc_accuracy'], 'EEGNet SC ITR': ts_eeg['sc_itr'],
        'BaseCNN MC Acc': ts_cnn['mc_accuracy'], 'BaseCNN MC ITR': ts_cnn['mc_itr'],
        'BaseCNN SC Acc': ts_cnn['sc_accuracy'], 'BaseCNN SC ITR': ts_cnn['sc_itr'],
        'SVM Acc': ts_svm['accuracy'], 'SVM ITR': ts_svm['itr'],
    })

# --- Display ---
df_unified = pd.DataFrame(rows)
pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 20)
display(df_unified)

# --- LaTeX export ---
latex_table = df_unified.to_latex(
    index=False,
    float_format="%.3f",
    caption="Unified comparison of all data aggregation approaches",
    label="tab:unified_comparison"
)
print(latex_table)